# Watermark Segmentation - Kaggle Training

This notebook trains **YOLO11n-seg** on the watermark dataset (`wm_dataset_out`), then evaluates and tests it on real scraped documents (`images_scraped`). Section 10 (below, after the YOLO sections) trains a second, independent model -- **the alpha network** (per-pixel watermark opacity).

### Dataset Layout in Kaggle

The Kaggle dataset is titled **"train and test2"**, mounted at `/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2` (older notebook runs may still see it at the shorter `/kaggle/input/train-and-test2` or `/kaggle/input/train-and-test` -- all three paths are checked below):

```
/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/
  images_scraped/images_scraped/*.jpg          # real test pages (no labels) -- section 8
  wm_dataset_colab/wm_dataset_out/
    annotations/  image_labels.csv, train.json, val.json   (COCO)
    images/{train,val}/wm_NNNN.jpg             # 1700 train / 300 val, all 1024x1024 -- sections 1-9
    labels/{train,val}/wm_NNNN.txt             # YOLO-seg, single class 0
    masks/{train,val}/wm_NNNN_mask.png         # class-encoded 0/1/2/3 masks
  wm_backgrounds_v2/                            # needed only by section 10 (alpha network)
  watermarks_processed/                         # needed only by section 10 (alpha network)
```

- Output directory (writable): `/kaggle/working`
- Sections 1-9 (YOLO segmentation) only need `wm_dataset_colab/wm_dataset_out` and `images_scraped/`.
- Section 10 (the alpha network) locates these same two folders itself (10.1) plus `wm_backgrounds_v2/` and `watermarks_processed/` at the top level of the dataset -- see section 10's markdown for what those two extra folders are needed for.


## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))


## 2. Install Ultralytics

In [ ]:
%pip -q install ultralytics==8.4.142
import ultralytics
ultralytics.checks()


## 3. Locate Dataset in Kaggle Input

Kaggle mounts datasets under `/kaggle/input/`. We locate `wm_dataset_out` and `images_scraped` automatically.

In [ ]:
import pathlib

# Primary path based on Kaggle dataset hierarchy
DATASET_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/wm_dataset_colab/wm_dataset_out'),
    pathlib.Path('/kaggle/input/train-and-test2/wm_dataset_colab/wm_dataset_out'),
    pathlib.Path('/kaggle/input/train-and-test/wm_dataset_colab/wm_dataset_out'),
    pathlib.Path('/kaggle/input/train-and-test/wm_dataset_out'),
]

ROOT = None
for cand in DATASET_CANDIDATES:
    if cand.exists():
        ROOT = cand
        break

if ROOT is None:
    # Recursive search if folder structure differs slightly -- prefer a
    # wm_dataset_out that actually contains images/train (an input could
    # in principle carry more than one folder with this name).
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('wm_dataset_out') if (p / 'images' / 'train').is_dir()]
    if not matches:
        matches = list(pathlib.Path('/kaggle/input').rglob('wm_dataset_out'))
    if matches:
        ROOT = matches[0]

assert ROOT and ROOT.exists(), f'Could not find wm_dataset_out in /kaggle/input. Found: {list(pathlib.Path("/kaggle/input").glob("*"))}'
print('Dataset ROOT found at:', ROOT)


## 4. Write `data.yaml` with ABSOLUTE paths

Ultralytics requires absolute paths to avoid resolving against internal default directories.

In [ ]:
import pathlib, yaml

data = {
    'path':  str(ROOT),          # ABSOLUTE path to dataset root
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': ['watermark'],
}
yaml_path = pathlib.Path('/kaggle/working/data.yaml')
yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
print(yaml_path.read_text())

from ultralytics.data.utils import check_det_dataset
resolved = check_det_dataset(str(yaml_path))
print('resolved train ->', resolved['train'])
print('resolved val   ->', resolved['val'])


## 5. Sanity-check the dataset before spending GPU time

Confirms every image has a matching label, and counts the intentional **negatives** (empty label files = clean pages with no watermark).

In [ ]:
for split in ['train', 'val']:
    imgs = sorted((ROOT/'images'/split).glob('*'))
    labs = sorted((ROOT/'labels'/split).glob('*.txt'))
    stems_i = {p.stem for p in imgs}
    stems_l = {p.stem for p in labs}
    empties = [p for p in labs if p.stat().st_size == 0]
    print(f'{split:5} images={len(imgs):4d} labels={len(labs):4d} '
          f'unmatched={len(stems_i ^ stems_l):3d} negatives={len(empties):3d}')
    assert not (stems_i ^ stems_l), f'{split}: image/label filenames do not match'


## 6. Train

1700 train / 300 val images (1024x1024, 2000 total). On a Kaggle T4 or P100 expect roughly 30-50 minutes.

`patience=30` early-stops if validation stops improving.
On CUDA OOM: drop `batch` to 8, then 4, and only reduce `imgsz` as a last resort.

In [ ]:
from ultralytics import YOLO
import pathlib
import shutil
from IPython.display import FileLink, display

# ==============================================================================
# Resume / Continue Training Configuration
# ==============================================================================
# Set RESUME_FROM_CHECKPOINT to:
#   - None: start fresh from yolo11n-seg.pt (COCO-pretrained)
#   - Path to existing best.pt or checkpoint, e.g. '/kaggle/working/runs/wm_seg/weights/best.pt',
#     an uploaded dataset weight (e.g. '/kaggle/input/.../best.pt'),
#     or a previous checkpoint like '/kaggle/working/best_epoch_30.pt'
CHECKPOINT_PATH = '/kaggle/working/runs/wm_seg/weights/best.pt'  # default check path
RESUME_FROM_CHECKPOINT = CHECKPOINT_PATH if pathlib.Path(CHECKPOINT_PATH).exists() else None

if RESUME_FROM_CHECKPOINT:
    print(f'Continuing training from checkpoint: {RESUME_FROM_CHECKPOINT}')
    model = YOLO(str(RESUME_FROM_CHECKPOINT))
else:
    print('Starting fresh training from yolo11n-seg.pt (COCO-pretrained)')
    model = YOLO('yolo11n-seg.pt')

def on_train_epoch_end_hook(trainer):
    # Every 10 epochs
    current_epoch = trainer.epoch + 1
    if current_epoch % 10 == 0:
        best_path = pathlib.Path(trainer.save_dir) / 'weights' / 'best.pt'
        if best_path.exists():
            out_file = pathlib.Path(f'/kaggle/working/best_epoch_{current_epoch}.pt')
            shutil.copy2(best_path, out_file)
            print(f'\n[Checkpoint] Saved best weights at epoch {current_epoch} to {out_file}')
            display(FileLink(f'best_epoch_{current_epoch}.pt'))

model.add_callback('on_train_epoch_end', on_train_epoch_end_hook)

results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=1024,      # thin-ring shield logo needs the resolution
    batch=16,
    patience=30,
    seed=0,
    project='/kaggle/working/runs',
    name='wm_seg',
    exist_ok=True,   # allow updating existing run directory if continuing
    plots=True,
)


### 6b. Variant - freeze the first half of the network

Section 6 trains **every** layer (`freeze` is unset, which is Ultralytics' default). This cell is the alternative: freeze the first half and update only the second half.

`freeze=N` counts **top-level modules, not parameters**. `yolo11n-seg` has 23 top-level modules (0..22 are the backbone + neck, 23 is the segment head). `freeze=11` freezes modules 0..10 and leaves 11..23 trainable.

In [ ]:
from ultralytics import YOLO

FREEZE   = 11                 # freeze modules 0..11, train 12..23
RUN_NAME = 'wm_seg_frozen'     # separate run dir, so section 6's weights survive

# Print exactly what will and will not be trained BEFORE spending GPU time.
_probe = YOLO('yolo11n-seg.pt')
_mods  = _probe.model.model
print(f'Total top-level modules: {len(_mods)}')
print(f'Frozen   (0..{FREEZE-1}): {[type(m).__name__ for m in _mods[:FREEZE]]}')
print(f'Trainable ({FREEZE}..{len(_mods)-1}): {[type(m).__name__ for m in _mods[FREEZE:]]}')
del _probe

model_frozen = YOLO('yolo11n-seg.pt')
results_frozen = model_frozen.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=1024,
    batch=16,
    patience=30,
    seed=0,
    project='/kaggle/working/runs',
    name=RUN_NAME,
    freeze=FREEZE,
    plots=True,
)


## 7. Validate and look at predictions

These mAP numbers are on **held-out synthetic** data.

In [ ]:
metrics = model.val()
print('mask mAP50    :', round(float(metrics.seg.map50), 4))
print('mask mAP50-95 :', round(float(metrics.seg.map), 4))
print('mask precision:', round(float(metrics.seg.mp), 4))
print('mask recall   :', round(float(metrics.seg.mr), 4))


In [ ]:
import glob
from IPython.display import Image as IPyImage, display

for p in sorted(glob.glob('/kaggle/working/runs/wm_seg/val_batch*_pred.jpg'))[:3]:
    print(p)
    display(IPyImage(filename=p, width=900))


## 8. Inference on the REAL test set (`images_scraped`)

This section runs inference on the real documents provided in the dataset input under `images_scraped`.

In [ ]:
import pathlib

# Locate images_scraped
TEST_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/images_scraped/images_scraped'),
    pathlib.Path('/kaggle/input/train-and-test2/images_scraped/images_scraped'),
    pathlib.Path('/kaggle/input/train-and-test/images_scraped/images_scraped'),
    pathlib.Path('/kaggle/input/train-and-test/images_scraped'),
]

test_path = None
for cand in TEST_CANDIDATES:
    if cand.exists() and any(cand.iterdir()):
        test_path = cand
        break

if test_path is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('images_scraped') if p.is_dir()]
    if matches:
        # Prefer the deepest subfolder if nested
        test_path = sorted(matches, key=lambda p: len(p.parts))[-1]

assert test_path and test_path.exists(), f'Could not find images_scraped in /kaggle/input'
print('Test folder found at:', test_path)

EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
test_images = sorted(p for p in test_path.rglob('*') if p.suffix.lower() in EXTS and p.is_file())
print(f'{test_path}  ->  {len(test_images)} images')
assert test_images, f'No images found under {test_path}'


### 8b. Run inference and render overlays

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from ultralytics import YOLO

CONF = 0.25
IMGSZ = 1024          # match training
MASK_COLOR = (255, 40, 40)
MASK_ALPHA = 0.45

best_w = pathlib.Path('/kaggle/working/runs/wm_seg/weights/best.pt')
assert best_w.exists(), f'No trained weights at {best_w} - run the training cell first.'
infer_model = YOLO(str(best_w))

OUT_DIR = pathlib.Path('/kaggle/working/test_overlays'); OUT_DIR.mkdir(exist_ok=True)
rows = []

for i, p in enumerate(test_images, 1):
    img = Image.open(p).convert('RGB')
    W, H = img.size
    r = infer_model.predict(source=str(p), conf=CONF, imgsz=IMGSZ, verbose=False)[0]

    masks = None
    if r.masks is not None and r.masks.data is not None and len(r.masks.data):
        m = r.masks.data.cpu().numpy()
        resized = [(np.asarray(Image.fromarray((s * 255).astype(np.uint8)).resize((W, H), Image.NEAREST)) > 127)
                   for s in m]
        masks = np.stack(resized) if resized else None
    confs = [float(c) for c in r.boxes.conf.cpu().numpy()] if (r.boxes is not None and len(r.boxes)) else []

    arr = np.asarray(img).astype(np.float32)
    if masks is not None:
        union = masks.sum(axis=0) > 0
        arr[union] = (1 - MASK_ALPHA) * arr[union] + MASK_ALPHA * np.array(MASK_COLOR, np.float32)
    overlaid = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    n_inst = 0 if masks is None else int(masks.shape[0])
    cov = 0.0 if masks is None else float((masks.sum(axis=0) > 0).mean())

    panel = Image.new('RGB', (W * 2 + 8, H + 26), (245, 245, 245))
    panel.paste(img, (0, 26)); panel.paste(overlaid, (W + 8, 26))
    d = ImageDraw.Draw(panel)
    d.text((4, 6), f'original - {p.name}', fill=(40, 40, 40))
    d.text((W + 12, 6), f'prediction: {n_inst} inst, {cov*100:.2f}% of page', fill=(180, 30, 30))
    panel.save(OUT_DIR / f'{p.stem}_overlay.png')

    rows.append({'image': p.name, 'instances': n_inst, 'coverage': cov,
                 'conf_max': max(confs) if confs else 0.0})
    if i % 20 == 0 or i == len(test_images):
        print(f'  [{i}/{len(test_images)}]')

n_any = sum(1 for r_ in rows if r_['instances'] > 0)
covs = [r_['coverage'] for r_ in rows]
inst = [r_['instances'] for r_ in rows]
print('\n' + '=' * 64)
print('DESCRIPTIVE ONLY - no ground truth, so these are NOT accuracy metrics')
print('=' * 64)
print(f'  images                    : {len(rows)}')
print(f'  images with >=1 detection : {n_any}  ({100*n_any/len(rows):.1f}%)')
print(f'  instances/image           : mean {np.mean(inst):.1f}  median {np.median(inst):.0f}  max {max(inst)}')
print(f'  page coverage             : mean {100*np.mean(covs):.2f}%  median {100*np.median(covs):.2f}%  max {100*max(covs):.2f}%')


### 8c. Look at the overlays

Red = predicted watermark pixels. Shows the first 12; all of them are saved to `/kaggle/working/test_overlays/`.

In [ ]:
from IPython.display import Image as IPyImage, display

SHOW = 12
for r_ in rows[:SHOW]:
    stem = pathlib.Path(r_['image']).stem
    print(f"{r_['image']}  ->  {r_['instances']} inst, {r_['coverage']*100:.2f}% coverage, "
          f"max conf {r_['conf_max']:.2f}")
    display(IPyImage(filename=str(OUT_DIR / f'{stem}_overlay.png'), width=900))


## 9. Model weights and output artifacts in Kaggle

All outputs (trained weights `best.pt`, plots, logs, and `test_overlays/`) are automatically saved to `/kaggle/working/`.
They will be available in the **Output** tab of your Kaggle notebook session.

In [ ]:
import pathlib

best = pathlib.Path('/kaggle/working/runs/wm_seg/weights/best.pt')
if best.exists():
    print('Trained model weights ready at:', best)
    print('File size:', round(best.stat().st_size / 1e6, 2), 'MB')
else:
    print('best.pt not found yet. Make sure training completed successfully.')


## 10. Alpha network (per-pixel watermark opacity)

This section trains a second, independent model: a small U-Net (`AlphaUNet`) that predicts, for every pixel of a page, how much semi-transparent watermark opacity ("alpha") sits there and what colour the mark's own ink is. Because the compositing physics is known --

    observed = alpha * ink + (1 - alpha) * true_page

-- the true page is recovered by the closed-form inverse `true = (observed - alpha*ink) / (1 - alpha)`, not by classification + fill/subtract like the Document tab's other methods.

Training needs, per image, the CLEAN (unwatermarked) page and the exact per-pixel opacity map -- neither is in the reference dataset directly, so 10.3 below rebuilds them by re-running the dataset generator with the same random seed against `wm_backgrounds_v2/` + `watermarks_processed/` (one run, ~23 minutes on a Kaggle kernel, ~2.2 GB written to `/kaggle/working`) and reading off the intermediate values as it goes.

Cell order: 10.1 locates the four dataset folders this section needs; 10.2 writes the generator and the replay tool to `/kaggle/working`; 10.3 rebuilds the training targets (skipped automatically once already done); 10.4 sanity-checks them; 10.5/10.6 define the model/dataset/loss; 10.7 is the actual training cell; 10.8/10.9 are validation visuals on held-out crops and on real scraped documents; 10.10 packages the final weights for download.


### 10.1 Locate the datasets

Locates the four folders this section needs under `/kaggle/input`: `wm_dataset_colab/wm_dataset_out` (already used by the YOLO sections above), `images_scraped`, and the two extra folders needed only here, `wm_backgrounds_v2` and `watermarks_processed`.


In [ ]:
import pathlib

# --- wm_dataset_out (images/labels/annotations for the YOLO sections; also
# used here to pair replayed alpha/clean targets with the reference JPEGs) ---
WM_DATASET_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/wm_dataset_colab/wm_dataset_out'),
    pathlib.Path('/kaggle/input/train-and-test2/wm_dataset_colab/wm_dataset_out'),
    pathlib.Path('/kaggle/input/train-and-test/wm_dataset_colab/wm_dataset_out'),
]
WM_DATASET = None
for cand in WM_DATASET_CANDIDATES:
    if cand.exists():
        WM_DATASET = cand
        break
if WM_DATASET is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('wm_dataset_out') if (p / 'images' / 'train').is_dir()]
    if matches:
        WM_DATASET = matches[0]
assert WM_DATASET and WM_DATASET.exists(), f'Could not find wm_dataset_out in /kaggle/input. Found: {list(pathlib.Path("/kaggle/input").glob("*"))}'
print('WM_DATASET      ->', WM_DATASET)

# --- images_scraped (real, unlabelled test pages) ---
SCRAPED_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/images_scraped/images_scraped'),
    pathlib.Path('/kaggle/input/train-and-test2/images_scraped/images_scraped'),
    pathlib.Path('/kaggle/input/train-and-test/images_scraped/images_scraped'),
]
SCRAPED_DIR = None
for cand in SCRAPED_CANDIDATES:
    if cand.exists() and any(cand.iterdir()):
        SCRAPED_DIR = cand
        break
if SCRAPED_DIR is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('images_scraped') if p.is_dir()]
    if matches:
        SCRAPED_DIR = sorted(matches, key=lambda p: len(p.parts))[-1]  # prefer the deepest (most specific)
print('SCRAPED_DIR     ->', SCRAPED_DIR)

# --- wm_backgrounds_v2 (2000 clean background images, replay input) ---
BACKGROUNDS_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/wm_backgrounds_v2'),
    pathlib.Path('/kaggle/input/train-and-test2/wm_backgrounds_v2'),
]
BACKGROUNDS_DIR = None
for cand in BACKGROUNDS_CANDIDATES:
    if cand.exists():
        BACKGROUNDS_DIR = cand
        break
if BACKGROUNDS_DIR is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('wm_backgrounds_v2') if p.is_dir()]
    if matches:
        BACKGROUNDS_DIR = matches[0]
print('BACKGROUNDS_DIR ->', BACKGROUNDS_DIR)

# --- watermarks_processed (3 watermark PNGs, replay input) ---
WATERMARKS_CANDIDATES = [
    pathlib.Path('/kaggle/input/datasets/amirrezayazdanpanah/train-and-test2/watermarks_processed'),
    pathlib.Path('/kaggle/input/train-and-test2/watermarks_processed'),
]
WATERMARKS_DIR = None
for cand in WATERMARKS_CANDIDATES:
    if cand.exists():
        WATERMARKS_DIR = cand
        break
if WATERMARKS_DIR is None:
    matches = [p for p in pathlib.Path('/kaggle/input').rglob('watermarks_processed') if p.is_dir()]
    if matches:
        WATERMARKS_DIR = matches[0]
print('WATERMARKS_DIR  ->', WATERMARKS_DIR)

# Where the rebuilt training targets / training run will live.
TARGETS_DIR = pathlib.Path('/kaggle/working/alpha_targets')
ALPHA_RUN_DIR = pathlib.Path('/kaggle/working/runs/alpha_net')


### 10.2 Write the dataset generator and the replay script

Both files are byte-identical to their repo copies (`scripts/kaggle_dataset/generate_dataset.py`, `scripts/kaggle_dataset/replay_alpha_targets.py`). Like `!nvidia-smi` above, these are tools invoked from the shell (10.3 runs the replay script as a subprocess) rather than modules imported into this kernel.


In [ ]:
%%writefile /kaggle/working/generate_dataset.py
"""
generate_dataset.py
Main dataset generation pipeline:
  1. Loads all backgrounds (2000 total) and 3 processed watermarks
  2. Assigns watermark combinations (0/1/2/3 watermarks per image)
  3. Applies extensive augmentations to watermarks
  4. Composites watermarks onto backgrounds
  5. Generates COCO JSON annotations, binary segmentation masks, and CSV labels
  6. Organizes into train/val splits (85/15)

Usage:
    python scripts/generate_dataset.py
    python scripts/generate_dataset.py --verify   # run verification checks
"""

import os
import sys
import json
import random
import math
import argparse
import csv
from datetime import datetime
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional

import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance, ImageChops
import cv2


# ─── Configuration ────────────────────────────────────────────────────────────

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
BG_DIR = os.path.join(BASE_DIR, "wm_backgrounds_v2")
WM_DIR = os.path.join(BASE_DIR, "watermarks_processed")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

TARGET_SIZE = (1024, 1024)  # All output images are 1024x1024
SEED = 42
TOTAL_IMAGES = 2000

# Watermark combination distribution
DIST_NO_WM = 0.10    # 10% = 200 images
DIST_1_WM = 0.40     # 40% = 800 images
DIST_2_WM = 0.30     # 30% = 600 images
DIST_3_WM = 0.20     # 20% = 400 images

# Split ratio
TRAIN_RATIO = 0.85    # 85% train, 15% val

# Augmentation ranges
OPACITY_RANGE = (0.10, 0.70)
ROTATION_RANGE = (-45, 45)
SCALE_RANGE = (0.15, 3.0)
BLUR_SIGMA_RANGE = (0, 3.0)
BRIGHTNESS_RANGE = (0.5, 1.5)
CONTRAST_RANGE = (0.5, 1.5)
PERSPECTIVE_SKEW = 5  # degrees
FEATHER_SIGMA_RANGE = (0, 5)
JPEG_QUALITY_RANGE = (50, 95)
NOISE_SIGMA_RANGE = (0, 10)

# Blend mode weights
BLEND_MODES = {
    "normal": 0.50,
    "multiply": 0.20,
    "screen": 0.20,
    "overlay": 0.10,
}

# Watermark names and IDs
WATERMARK_INFO = {
    "AriaTendernet": {"id": 1, "file": "AriaTendernet.png"},
    "ariatender_logo": {"id": 2, "file": "ariatender_logo.png"},
    "ariatender_persian": {"id": 3, "file": "ariatender_persian.png"},
}

CATEGORIES = [
    {"id": 0, "name": "no_watermark", "supercategory": "background"},
    {"id": 1, "name": "AriaTendernet", "supercategory": "watermark"},
    {"id": 2, "name": "ariatender_logo", "supercategory": "watermark"},
    {"id": 3, "name": "ariatender_persian", "supercategory": "watermark"},
]


# ─── Utility Functions ───────────────────────────────────────────────────────

def load_backgrounds() -> List[str]:
    """Load all background image paths."""
    files = sorted([
        os.path.join(BG_DIR, f)
        for f in os.listdir(BG_DIR)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])
    print(f"Found {len(files)} background images")
    return files


def load_watermarks() -> Dict[str, Image.Image]:
    """Load all processed watermark images."""
    wms = {}
    for name, info in WATERMARK_INFO.items():
        path = os.path.join(WM_DIR, info["file"])
        wm = Image.open(path).convert("RGBA")
        wms[name] = wm
        print(f"  Loaded watermark '{name}': {wm.size}")
    return wms


def choose_blend_mode() -> str:
    """Randomly choose a blend mode based on weights."""
    modes = list(BLEND_MODES.keys())
    weights = list(BLEND_MODES.values())
    return random.choices(modes, weights=weights, k=1)[0]


def apply_blend(base_rgb: np.ndarray, wm_rgb: np.ndarray, mode: str) -> np.ndarray:
    """
    Apply blending between base (background under watermark) and watermark RGB.
    Both inputs are float [0, 1]. Returns blended RGB [0, 1].
    """
    if mode == "normal":
        return wm_rgb
    elif mode == "multiply":
        return base_rgb * wm_rgb
    elif mode == "screen":
        return 1.0 - (1.0 - base_rgb) * (1.0 - wm_rgb)
    elif mode == "overlay":
        # Overlay: multiply if base < 0.5, screen if base >= 0.5
        mask = base_rgb < 0.5
        result = np.where(
            mask,
            2.0 * base_rgb * wm_rgb,
            1.0 - 2.0 * (1.0 - base_rgb) * (1.0 - wm_rgb)
        )
        return result
    return wm_rgb


def apply_perspective_warp(img: Image.Image, max_skew_deg: float = 5) -> Image.Image:
    """Apply a slight perspective/affine warp to the image."""
    w, h = img.size
    
    # Random skew amounts in pixels
    max_offset = int(max(w, h) * math.tan(math.radians(max_skew_deg)) * 0.1)
    if max_offset < 1:
        return img
    
    # Define source and destination corners
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = np.float32([
        [random.randint(0, max_offset), random.randint(0, max_offset)],
        [w - random.randint(0, max_offset), random.randint(0, max_offset)],
        [w - random.randint(0, max_offset), h - random.randint(0, max_offset)],
        [random.randint(0, max_offset), h - random.randint(0, max_offset)],
    ])
    
    # Convert to numpy for cv2
    img_np = np.array(img)
    M = cv2.getPerspectiveTransform(src, dst)
    result = cv2.warpPerspective(img_np, M, (w, h), borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0, 0))
    
    return Image.fromarray(result)


def apply_color_jitter(img: Image.Image) -> Image.Image:
    """Apply slight color jitter (hue/saturation shift) to watermark."""
    # Only jitter sometimes
    if random.random() > 0.3:
        return img
    
    # Split channels
    r, g, b, a = img.split()
    rgb = Image.merge("RGB", (r, g, b))
    
    # Slight hue shift via HSV
    hsv = np.array(rgb.convert("HSV"), dtype=np.float32)
    hsv[:, :, 0] = (hsv[:, :, 0] + random.uniform(-10, 10)) % 180
    hsv[:, :, 1] = np.clip(hsv[:, :, 1] * random.uniform(0.8, 1.2), 0, 255)
    hsv = hsv.astype(np.uint8)
    
    rgb_shifted = Image.fromarray(hsv, "HSV").convert("RGB")
    return Image.merge("RGBA", (*rgb_shifted.split(), a))


def augment_watermark(
    wm: Image.Image,
    bg_size: Tuple[int, int],
) -> Tuple[Image.Image, dict]:
    """
    Apply all augmentations to a single watermark instance.
    Returns the augmented watermark (RGBA) and a dict of augmentation parameters.
    """
    params = {}
    
    # 1. Scale
    scale = random.uniform(*SCALE_RANGE)
    # Bias toward moderate scales (0.3-1.5) with 70% probability
    if random.random() < 0.7:
        scale = random.uniform(0.3, 1.5)
    params["scale"] = round(scale, 3)
    
    new_w = max(1, int(wm.width * scale))
    new_h = max(1, int(wm.height * scale))
    # Cap to prevent memory issues
    new_w = min(new_w, bg_size[0] * 4)
    new_h = min(new_h, bg_size[1] * 4)
    wm_aug = wm.resize((new_w, new_h), Image.LANCZOS)
    
    # 2. Rotation
    rotation = random.uniform(*ROTATION_RANGE)
    # Bias toward common angles
    if random.random() < 0.4:
        rotation = random.choice([0, 0, 0, -45, 45, -30, 30, -15, 15])
    params["rotation"] = round(rotation, 1)
    wm_aug = wm_aug.rotate(rotation, expand=True, resample=Image.BICUBIC, fillcolor=(0, 0, 0, 0))
    
    # 3. Resolution/blur
    blur_sigma = random.uniform(*BLUR_SIGMA_RANGE)
    if blur_sigma > 0.3:
        wm_aug = wm_aug.filter(ImageFilter.GaussianBlur(radius=blur_sigma))
    params["blur_sigma"] = round(blur_sigma, 2)
    
    # 4. Brightness
    brightness = random.uniform(*BRIGHTNESS_RANGE)
    params["brightness"] = round(brightness, 2)
    r, g, b, a = wm_aug.split()
    rgb = Image.merge("RGB", (r, g, b))
    rgb = ImageEnhance.Brightness(rgb).enhance(brightness)
    wm_aug = Image.merge("RGBA", (*rgb.split(), a))
    
    # 5. Contrast
    contrast = random.uniform(*CONTRAST_RANGE)
    params["contrast"] = round(contrast, 2)
    r, g, b, a = wm_aug.split()
    rgb = Image.merge("RGB", (r, g, b))
    rgb = ImageEnhance.Contrast(rgb).enhance(contrast)
    wm_aug = Image.merge("RGBA", (*rgb.split(), a))
    
    # 6. Color jitter
    wm_aug = apply_color_jitter(wm_aug)
    
    # 7. Opacity
    opacity = random.uniform(*OPACITY_RANGE)
    params["opacity"] = round(opacity, 2)
    # We'll apply opacity during compositing, not here
    
    # 8. Perspective warp (30% of the time)
    if random.random() < 0.3:
        wm_aug = apply_perspective_warp(wm_aug, PERSPECTIVE_SKEW)
        params["perspective_warp"] = True
    else:
        params["perspective_warp"] = False
    
    # 9. Edge feathering (40% of the time)
    feather = random.uniform(*FEATHER_SIGMA_RANGE)
    if feather > 0.5 and random.random() < 0.4:
        r, g, b, a = wm_aug.split()
        a = a.filter(ImageFilter.GaussianBlur(radius=feather))
        wm_aug = Image.merge("RGBA", (r, g, b, a))
        params["edge_feather"] = round(feather, 2)
    else:
        params["edge_feather"] = 0
    
    # 10. Blend mode
    blend_mode = choose_blend_mode()
    params["blend_mode"] = blend_mode
    
    return wm_aug, params


def decide_tiling(bg_size: Tuple[int, int]) -> Tuple[bool, int, int, int, int]:
    """Decide if and how to tile the watermark."""
    # 20% chance of tiling
    if random.random() < 0.2:
        cols = random.randint(2, 5)
        rows = random.randint(2, 5)
        gap_x = random.randint(20, 150)
        gap_y = random.randint(20, 100)
        return True, rows, cols, gap_x, gap_y
    return False, 1, 1, 0, 0


def composite_watermark_on_background(
    bg: Image.Image,
    wm_aug: Image.Image,
    opacity: float,
    blend_mode: str,
    position: Tuple[int, int],
) -> Tuple[Image.Image, Image.Image]:
    """
    Composite a single augmented watermark onto the background.
    Returns (composited_bg, mask_layer).
    
    mask_layer is a single-channel image where non-zero pixels indicate watermark presence.
    """
    bg_w, bg_h = bg.size
    wm_w, wm_h = wm_aug.size
    px, py = position
    
    # Create the mask layer (same size as bg)
    mask_layer = Image.new("L", (bg_w, bg_h), 0)
    
    # Calculate the visible region (intersection of watermark and background)
    # Watermark region in bg coordinates
    x1 = max(0, px)
    y1 = max(0, py)
    x2 = min(bg_w, px + wm_w)
    y2 = min(bg_h, py + wm_h)
    
    if x2 <= x1 or y2 <= y1:
        return bg, mask_layer  # No overlap
    
    # Crop the watermark to the visible region
    wm_x1 = x1 - px
    wm_y1 = y1 - py
    wm_x2 = x2 - px
    wm_y2 = y2 - py
    
    wm_crop = wm_aug.crop((wm_x1, wm_y1, wm_x2, wm_y2))
    
    if wm_crop.size[0] == 0 or wm_crop.size[1] == 0:
        return bg, mask_layer
    
    # Extract channels
    wm_r, wm_g, wm_b, wm_a = wm_crop.split()
    
    # Apply opacity to alpha
    wm_a_np = np.array(wm_a, dtype=np.float32) / 255.0
    wm_a_np = wm_a_np * opacity
    
    # Get the background region
    bg_region = bg.crop((x1, y1, x2, y2)).convert("RGB")
    bg_np = np.array(bg_region, dtype=np.float32) / 255.0
    
    # Get watermark RGB
    wm_rgb_np = np.stack([
        np.array(wm_r, dtype=np.float32) / 255.0,
        np.array(wm_g, dtype=np.float32) / 255.0,
        np.array(wm_b, dtype=np.float32) / 255.0,
    ], axis=-1)
    
    # Apply blend mode
    blended = apply_blend(bg_np, wm_rgb_np, blend_mode)
    
    # Alpha composite: result = bg * (1 - alpha) + blended * alpha
    alpha_3ch = np.stack([wm_a_np] * 3, axis=-1)
    result = bg_np * (1.0 - alpha_3ch) + blended * alpha_3ch
    result = np.clip(result * 255, 0, 255).astype(np.uint8)
    
    # Paste result back
    bg_result = bg.copy()
    bg_result.paste(Image.fromarray(result), (x1, y1))
    
    # Build mask: where alpha > threshold
    mask_region = (wm_a_np > 0.02).astype(np.uint8) * 255
    mask_layer.paste(Image.fromarray(mask_region, "L"), (x1, y1))
    
    return bg_result, mask_layer


def apply_post_processing(img: Image.Image) -> Image.Image:
    """Apply post-composition augmentations to the final image."""
    # Convert to numpy for some ops
    img_np = np.array(img, dtype=np.float32)
    
    # 1. Gaussian noise (60% of the time)
    if random.random() < 0.6:
        noise_sigma = random.uniform(*NOISE_SIGMA_RANGE)
        if noise_sigma > 1:
            noise = np.random.normal(0, noise_sigma, img_np.shape)
            img_np = np.clip(img_np + noise, 0, 255)
    
    # 2. Color temperature shift (30% of the time)
    if random.random() < 0.3:
        shift = random.uniform(-15, 15)
        img_np[:, :, 0] = np.clip(img_np[:, :, 0] + shift, 0, 255)      # R channel
        img_np[:, :, 2] = np.clip(img_np[:, :, 2] - shift * 0.7, 0, 255)  # B channel
    
    img = Image.fromarray(img_np.astype(np.uint8))
    
    # 3. Background quality degradation (resize down then up) - 20% of the time
    if random.random() < 0.2:
        factor = random.uniform(0.4, 0.8)
        small_size = (max(64, int(img.width * factor)), max(64, int(img.height * factor)))
        img = img.resize(small_size, Image.BILINEAR).resize(img.size, Image.BILINEAR)
    
    return img


def save_as_jpeg_with_quality(img: Image.Image, path: str) -> None:
    """Save image as JPEG with random quality to simulate compression artifacts."""
    quality = random.randint(*JPEG_QUALITY_RANGE)
    img.convert("RGB").save(path, "JPEG", quality=quality)


def generate_single_image(
    idx: int,
    bg_path: str,
    watermarks: Dict[str, Image.Image],
    wm_combo: List[str],  # list of watermark names to apply
) -> dict:
    """
    Generate a single dataset image with watermarks and masks.
    Returns annotation metadata dict.
    """
    # Load and resize background to 1024x1024
    bg = Image.open(bg_path).convert("RGB")
    bg = bg.resize(TARGET_SIZE, Image.LANCZOS)
    
    bg_w, bg_h = bg.size
    
    # Determine output filename
    img_filename = f"wm_{idx:04d}.jpg"
    mask_filename = f"wm_{idx:04d}_mask.png"
    
    # Track annotations for this image
    annotations = []
    mask_layers = {}  # category_id -> combined mask
    
    if len(wm_combo) == 0:
        # No watermark — just save the clean image with post-processing
        bg = apply_post_processing(bg)
    else:
        for wm_name in wm_combo:
            wm_src = watermarks[wm_name].copy()
            cat_id = WATERMARK_INFO[wm_name]["id"]
            
            # Decide tiling
            is_tiled, tile_rows, tile_cols, gap_x, gap_y = decide_tiling((bg_w, bg_h))
            
            # Augment the watermark
            wm_aug, aug_params = augment_watermark(wm_src, (bg_w, bg_h))
            opacity = aug_params["opacity"]
            blend_mode = aug_params["blend_mode"]
            
            if is_tiled:
                # Place in a grid pattern
                wm_w, wm_h = wm_aug.size
                total_w = tile_cols * wm_w + (tile_cols - 1) * gap_x
                total_h = tile_rows * wm_h + (tile_rows - 1) * gap_y
                
                # Center the grid on the image
                start_x = (bg_w - total_w) // 2
                start_y = (bg_h - total_h) // 2
                
                combined_mask = Image.new("L", (bg_w, bg_h), 0)
                
                for r in range(tile_rows):
                    for c in range(tile_cols):
                        px = start_x + c * (wm_w + gap_x)
                        py = start_y + r * (wm_h + gap_y)
                        bg, ml = composite_watermark_on_background(
                            bg, wm_aug, opacity, blend_mode, (px, py)
                        )
                        # Merge mask
                        combined_mask = Image.fromarray(
                            np.maximum(np.array(combined_mask), np.array(ml))
                        )
                
                mask_layers[cat_id] = combined_mask
                aug_params["is_tiled"] = True
                aug_params["tile_grid"] = f"{tile_rows}x{tile_cols}"
                
                # Bounding box = extent of all tiles
                mask_np = np.array(combined_mask)
                if mask_np.max() > 0:
                    ys, xs = np.where(mask_np > 0)
                    bbox = [int(xs.min()), int(ys.min()),
                            int(xs.max() - xs.min()), int(ys.max() - ys.min())]
                    area = int(mask_np.sum() / 255)
                    is_partial = (bbox[0] <= 0 or bbox[1] <= 0 or
                                  bbox[0] + bbox[2] >= bg_w - 1 or
                                  bbox[1] + bbox[3] >= bg_h - 1)
                    
                    # Generate polygon from mask (simplified contour)
                    contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    segmentation = []
                    for contour in contours:
                        if len(contour) >= 3:
                            # Simplify contour
                            epsilon = 0.005 * cv2.arcLength(contour, True)
                            approx = cv2.approxPolyDP(contour, epsilon, True)
                            if len(approx) >= 3:
                                seg = approx.flatten().tolist()
                                segmentation.append(seg)
                    
                    annotations.append({
                        "category_id": cat_id,
                        "bbox": bbox,
                        "segmentation": segmentation,
                        "area": area,
                        "iscrowd": 0,
                        "attributes": {
                            "watermark_name": wm_name,
                            "is_partial": is_partial,
                            **aug_params,
                        }
                    })
            else:
                # Single placement
                wm_w, wm_h = wm_aug.size
                
                # Random position (allow partial off-screen)
                margin = 50
                px = random.randint(-wm_w + margin, bg_w - margin)
                py = random.randint(-wm_h + margin, bg_h - margin)
                
                bg, mask_layer = composite_watermark_on_background(
                    bg, wm_aug, opacity, blend_mode, (px, py)
                )
                mask_layers[cat_id] = mask_layer
                aug_params["is_tiled"] = False
                
                # Compute bounding box from mask
                mask_np = np.array(mask_layer)
                if mask_np.max() > 0:
                    ys, xs = np.where(mask_np > 0)
                    bbox = [int(xs.min()), int(ys.min()),
                            int(xs.max() - xs.min()), int(ys.max() - ys.min())]
                    area = int(mask_np.sum() / 255)
                    is_partial = (bbox[0] <= 0 or bbox[1] <= 0 or
                                  bbox[0] + bbox[2] >= bg_w - 1 or
                                  bbox[1] + bbox[3] >= bg_h - 1)
                    
                    # Generate polygon
                    contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    segmentation = []
                    for contour in contours:
                        if len(contour) >= 3:
                            epsilon = 0.005 * cv2.arcLength(contour, True)
                            approx = cv2.approxPolyDP(contour, epsilon, True)
                            if len(approx) >= 3:
                                seg = approx.flatten().tolist()
                                segmentation.append(seg)
                    
                    annotations.append({
                        "category_id": cat_id,
                        "bbox": bbox,
                        "segmentation": segmentation,
                        "area": area,
                        "iscrowd": 0,
                        "attributes": {
                            "watermark_name": wm_name,
                            "is_partial": is_partial,
                            **aug_params,
                        }
                    })
        
        # Post-processing
        bg = apply_post_processing(bg)
    
    # Build combined segmentation mask (class-encoded)
    # 0 = background, 1/2/3 = watermark class
    combined_mask = np.zeros((bg_h, bg_w), dtype=np.uint8)
    for cat_id, ml in mask_layers.items():
        ml_np = np.array(ml)
        combined_mask[ml_np > 0] = cat_id
    
    # Infer background type from filename
    bg_basename = os.path.basename(bg_path)
    
    return {
        "image": bg,
        "mask": combined_mask,
        "img_filename": img_filename,
        "mask_filename": mask_filename,
        "annotations": annotations,
        "bg_filename": bg_basename,
        "wm_combo": wm_combo,
        "has_watermark": len(wm_combo) > 0,
    }


def build_coco_dataset(
    all_results: List[dict],
    split_name: str,
) -> dict:
    """Build a COCO-format annotation dict from results."""
    images_list = []
    annotations_list = []
    ann_id = 1
    
    for r in all_results:
        img_id = int(r["img_filename"].split("_")[1].split(".")[0])
        images_list.append({
            "id": img_id,
            "file_name": r["img_filename"],
            "width": TARGET_SIZE[0],
            "height": TARGET_SIZE[1],
        })
        
        for ann in r["annotations"]:
            ann_entry = {
                "id": ann_id,
                "image_id": img_id,
                "category_id": ann["category_id"],
                "bbox": ann["bbox"],
                "segmentation": ann["segmentation"],
                "area": ann["area"],
                "iscrowd": ann["iscrowd"],
                "attributes": ann.get("attributes", {}),
            }
            annotations_list.append(ann_entry)
            ann_id += 1
    
    return {
        "info": {
            "description": f"Watermark Detection Dataset - {split_name}",
            "version": "1.0",
            "year": 2026,
            "date_created": datetime.now().isoformat(),
        },
        "images": images_list,
        "annotations": annotations_list,
        "categories": CATEGORIES,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--verify", action="store_true", help="Run verification checks only")
    args = parser.parse_args()
    
    if args.verify:
        run_verification()
        return
    
    random.seed(SEED)
    np.random.seed(SEED)
    
    print("=" * 70)
    print("  WATERMARK DATASET GENERATOR")
    print("=" * 70)
    print(f"  Target: {TOTAL_IMAGES} images @ {TARGET_SIZE[0]}x{TARGET_SIZE[1]}")
    print(f"  Split: {TRAIN_RATIO*100:.0f}% train / {(1-TRAIN_RATIO)*100:.0f}% val")
    print()
    
    # ─── Load assets ──────────────────────────────────────────────────────
    print("Loading backgrounds...")
    bg_paths = load_backgrounds()
    
    if len(bg_paths) < TOTAL_IMAGES:
        print(f"WARNING: Only {len(bg_paths)} backgrounds found, need {TOTAL_IMAGES}.")
        print("Will reuse backgrounds with different augmentations.")
        # Extend by cycling
        while len(bg_paths) < TOTAL_IMAGES:
            bg_paths.extend(bg_paths[:TOTAL_IMAGES - len(bg_paths)])
        bg_paths = bg_paths[:TOTAL_IMAGES]
    else:
        bg_paths = bg_paths[:TOTAL_IMAGES]
    
    random.shuffle(bg_paths)
    
    print("\nLoading watermarks...")
    watermarks = load_watermarks()
    wm_names = list(watermarks.keys())
    
    # ─── Assign watermark combinations ────────────────────────────────────
    print("\nAssigning watermark combinations...")
    n_no_wm = int(TOTAL_IMAGES * DIST_NO_WM)
    n_1_wm = int(TOTAL_IMAGES * DIST_1_WM)
    n_2_wm = int(TOTAL_IMAGES * DIST_2_WM)
    n_3_wm = TOTAL_IMAGES - n_no_wm - n_1_wm - n_2_wm
    
    combos = []
    # No watermark
    combos.extend([[] for _ in range(n_no_wm)])
    # 1 watermark
    for _ in range(n_1_wm):
        combos.append([random.choice(wm_names)])
    # 2 watermarks
    for _ in range(n_2_wm):
        combos.append(random.sample(wm_names, 2))
    # 3 watermarks
    combos.extend([list(wm_names) for _ in range(n_3_wm)])
    
    random.shuffle(combos)
    
    combo_counts = Counter([len(c) for c in combos])
    print(f"  No watermark: {combo_counts[0]}")
    print(f"  1 watermark:  {combo_counts[1]}")
    print(f"  2 watermarks: {combo_counts[2]}")
    print(f"  3 watermarks: {combo_counts[3]}")
    
    # ─── Create output directories ────────────────────────────────────────
    for split in ["train", "val"]:
        os.makedirs(os.path.join(OUTPUT_DIR, "images", split), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR, "masks", split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, "annotations"), exist_ok=True)
    
    # ─── Split assignments ────────────────────────────────────────────────
    indices = list(range(TOTAL_IMAGES))
    random.shuffle(indices)
    n_train = int(TOTAL_IMAGES * TRAIN_RATIO)
    train_indices = set(indices[:n_train])
    val_indices = set(indices[n_train:])
    
    print(f"\n  Train: {len(train_indices)} images")
    print(f"  Val:   {len(val_indices)} images")
    
    # ─── Generate images ──────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("  GENERATING DATASET")
    print(f"{'='*70}\n")
    
    train_results = []
    val_results = []
    csv_rows = []
    
    for idx in range(TOTAL_IMAGES):
        try:
            result = generate_single_image(
                idx, bg_paths[idx], watermarks, combos[idx]
            )
            
            # Determine split
            split = "train" if idx in train_indices else "val"
            
            # Save image (JPEG with quality augmentation)
            img_path = os.path.join(OUTPUT_DIR, "images", split, result["img_filename"])
            save_as_jpeg_with_quality(result["image"], img_path)
            
            # Save mask (PNG, lossless)
            mask_path = os.path.join(OUTPUT_DIR, "masks", split, result["mask_filename"])
            Image.fromarray(result["mask"]).save(mask_path, "PNG")
            
            # Track for COCO JSON
            if split == "train":
                train_results.append(result)
            else:
                val_results.append(result)
            
            # CSV row
            wm_types = ";".join(result["wm_combo"]) if result["wm_combo"] else ""
            csv_rows.append({
                "image_name": result["img_filename"],
                "split": split,
                "has_watermark": result["has_watermark"],
                "num_watermarks": len(result["wm_combo"]),
                "watermark_types": wm_types,
                "background_source": result["bg_filename"],
            })
            
            if (idx + 1) % 50 == 0:
                print(f"  [{idx+1:4d}/{TOTAL_IMAGES}] Generated {result['img_filename']} "
                      f"({split}, {len(result['wm_combo'])} wm)")
        
        except Exception as e:
            print(f"  ERROR generating image {idx}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # ─── Save COCO JSON annotations ──────────────────────────────────────
    print("\nSaving COCO JSON annotations...")
    
    train_coco = build_coco_dataset(train_results, "train")
    val_coco = build_coco_dataset(val_results, "val")
    
    train_json_path = os.path.join(OUTPUT_DIR, "annotations", "train.json")
    val_json_path = os.path.join(OUTPUT_DIR, "annotations", "val.json")
    
    with open(train_json_path, "w", encoding="utf-8") as f:
        json.dump(train_coco, f, indent=2, ensure_ascii=False)
    print(f"  Saved: {train_json_path} ({len(train_coco['annotations'])} annotations)")
    
    with open(val_json_path, "w", encoding="utf-8") as f:
        json.dump(val_coco, f, indent=2, ensure_ascii=False)
    print(f"  Saved: {val_json_path} ({len(val_coco['annotations'])} annotations)")
    
    # ─── Save CSV labels ─────────────────────────────────────────────────
    csv_path = os.path.join(OUTPUT_DIR, "annotations", "image_labels.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "image_name", "split", "has_watermark", "num_watermarks",
            "watermark_types", "background_source"
        ])
        writer.writeheader()
        writer.writerows(csv_rows)
    print(f"  Saved: {csv_path} ({len(csv_rows)} rows)")
    
    # ─── Summary ─────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("  GENERATION COMPLETE")
    print(f"{'='*70}")
    print(f"  Total images: {len(csv_rows)}")
    print(f"  Train: {len(train_results)}, Val: {len(val_results)}")
    print(f"  Output directory: {OUTPUT_DIR}")
    print()
    
    # Print distribution stats
    wm_count_dist = Counter(r["num_watermarks"] for r in csv_rows)
    print("  Watermark count distribution:")
    for k in sorted(wm_count_dist.keys()):
        print(f"    {k} watermarks: {wm_count_dist[k]}")
    print()


def run_verification():
    """Run verification checks on the generated dataset."""
    print("=" * 70)
    print("  DATASET VERIFICATION")
    print("=" * 70)
    
    errors = []
    
    # Check image counts
    for split in ["train", "val"]:
        img_dir = os.path.join(OUTPUT_DIR, "images", split)
        mask_dir = os.path.join(OUTPUT_DIR, "masks", split)
        
        if not os.path.exists(img_dir):
            errors.append(f"Missing directory: {img_dir}")
            continue
        
        imgs = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
        masks = [f for f in os.listdir(mask_dir) if f.endswith('.png')]
        
        print(f"\n  {split}: {len(imgs)} images, {len(masks)} masks")
        
        # Check mask-image correspondence
        img_bases = {os.path.splitext(f)[0] for f in imgs}
        mask_bases = {f.replace("_mask.png", "") for f in masks}
        
        missing_masks = img_bases - mask_bases
        if missing_masks:
            errors.append(f"{split}: {len(missing_masks)} images missing masks")
        
        # Check mask dimensions
        for mask_file in masks[:10]:  # spot check
            mask = Image.open(os.path.join(mask_dir, mask_file))
            if mask.size != TARGET_SIZE:
                errors.append(f"{split}/{mask_file}: mask size {mask.size} != {TARGET_SIZE}")
    
    # Check COCO JSON
    for split in ["train", "val"]:
        json_path = os.path.join(OUTPUT_DIR, "annotations", f"{split}.json")
        if not os.path.exists(json_path):
            errors.append(f"Missing annotation file: {json_path}")
            continue
        
        with open(json_path, "r") as f:
            coco = json.load(f)
        
        print(f"\n  {split}.json: {len(coco['images'])} images, {len(coco['annotations'])} annotations")
        print(f"  Categories: {[c['name'] for c in coco['categories']]}")
    
    # Check CSV
    csv_path = os.path.join(OUTPUT_DIR, "annotations", "image_labels.csv")
    if os.path.exists(csv_path):
        with open(csv_path, "r") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
        print(f"\n  image_labels.csv: {len(rows)} rows")
        
        wm_dist = Counter(int(r["num_watermarks"]) for r in rows)
        print("  Watermark distribution:")
        for k in sorted(wm_dist.keys()):
            print(f"    {k} watermarks: {wm_dist[k]}")
    
    # Report
    if errors:
        print(f"\n  [FAIL] VERIFICATION FAILED: {len(errors)} errors")
        for e in errors:
            print(f"    - {e}")
    else:
        print(f"\n  [SUCCESS] VERIFICATION PASSED")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/replay_alpha_targets.py
"""Deterministic replay of ``generate_dataset.py`` to recover EXACT
per-pixel alpha-regression targets (``clean``, per-pixel accumulated
watermark opacity ``A``) for the Kaggle "train and test2" dataset, whose
reference JPEGs/masks give no such ground truth on their own (see
``scripts/kaggle_dataset/README.md`` for why).

STANDALONE by design: stdlib + numpy + PIL + cv2 only, no repo imports.

How it works
------------
``generate_dataset.py`` seeds ``random``/``np.random`` with a fixed seed
and then draws every augmentation parameter (background shuffle, watermark
combo assignment, scale/rotation/blur/opacity/blend-mode/perspective/
feather/tiling/JPEG quality/noise/colour-shift/resize) from those two
global generators, in a fixed call order. If we run the *exact same file*
against the *exact same input assets* with the *exact same library
versions*, every draw reproduces bit-for-bit, so we can recover the exact
per-image compositing that produced each reference JPEG -- including the
otherwise-unrecorded per-pixel alpha and the pre-post-processing clean
background -- by monkeypatching four of its functions to *also* carry a
parallel "clean" image and an alpha accumulator ``A`` through the same
sequence of operations, using ONLY parameters that are already fixed by
the point they're read (no extra ``random``/``np.random`` calls are ever
added, removed, or reordered by the patches -- see the exactness checks
below, which are what prove that promise held).

This script imports the generator module fresh via ``importlib`` (so it
never pollutes ``sys.modules`` with a stale copy across repeated runs in
the same interpreter... except it does register under a fixed name --
re-running this script's ``run_replay`` twice in one process reloads the
module each time, which is fine since generator functions are pure
w.r.t. the patches installed just before each run), points its
``BG_DIR``/``WM_DIR``/``OUTPUT_DIR`` globals at the given paths, and
calls its unmodified ``main()``.

Exactness proof, per verified image (every ``--verify-every`` images):
JPEG-encode the replayed composite in memory at the SAME quality
``generate_dataset.py`` drew for it, decode it back, and compare pixel-
for-pixel against the reference JPEG decoded the same way. Only an
environment whose Pillow/libjpeg encoder is bit-identical to whatever
produced the reference dataset will show ``exact: true`` for every
verified image -- see the module docstring's honesty note and this run's
``replay_report.json`` for which local Python actually achieves that.
"""

from __future__ import annotations

import argparse
import csv
import importlib.util
import io
import json
import math
import random
import shutil
import sys
import time
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

try:
    import PIL
    _PIL_VERSION = getattr(PIL, "__version__", "unknown")
except Exception:  # pragma: no cover
    _PIL_VERSION = "unknown"


class _StopReplay(BaseException):
    """Raised (as a BaseException, NOT Exception) once ``--limit`` images
    have been produced, so it escapes the generator's own
    ``except Exception: continue`` inside ``main()``'s per-image loop
    instead of being silently swallowed there -- see module docstring.
    Never caught anywhere except around the ``main()`` call in
    ``run_replay``.
    """


# ---------------------------------------------------------------------------
# Generator module loading
# ---------------------------------------------------------------------------

def _load_generator_module(path: Path):
    spec = importlib.util.spec_from_file_location("kaggle_replay_generator", str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    return mod


# ---------------------------------------------------------------------------
# Histogram-based percentile (memory-safe over arbitrarily many images)
# ---------------------------------------------------------------------------

def _percentile_from_hist(hist_counts: np.ndarray, bin_edges: np.ndarray, q: float) -> float:
    total = int(hist_counts.sum())
    if total == 0:
        return 0.0
    target = q / 100.0 * total
    cum = np.cumsum(hist_counts)
    idx = int(np.searchsorted(cum, target, side="left"))
    idx = min(idx, len(bin_edges) - 2)
    return float(bin_edges[idx + 1])


# ---------------------------------------------------------------------------
# Core replay
# ---------------------------------------------------------------------------

def run_replay(args) -> dict:
    out_root = Path(args.out)
    gen_out_dir = out_root / "_replay_generator_out"
    clean_root = out_root / "clean"
    alpha_root = out_root / "alpha"

    if args.overwrite and out_root.exists():
        shutil.rmtree(out_root)

    for split in ("train", "val"):
        (clean_root / split).mkdir(parents=True, exist_ok=True)
        (alpha_root / split).mkdir(parents=True, exist_ok=True)
    gen_out_dir.mkdir(parents=True, exist_ok=True)

    gen_path = Path(args.generator) if args.generator else (Path(__file__).parent / "generate_dataset.py")
    mod = _load_generator_module(gen_path)
    mod.BG_DIR = str(Path(args.backgrounds_dir))
    mod.WM_DIR = str(Path(args.watermarks_dir))
    mod.OUTPUT_DIR = str(gen_out_dir)

    orig_composite = mod.composite_watermark_on_background
    orig_gen_single = mod.generate_single_image

    limit = args.limit
    state = {"clean": None, "A": None}
    captured_results = []  # one dict per successfully-generated image, in idx order
    per_image_records = []  # verification / stats records, one per image, same order

    hist_counts = np.zeros(256, dtype=np.int64)
    bin_edges = np.linspace(0.0, 1.0, 257)
    stats_acc = {"total_pixels": 0, "total_pos_pixels": 0, "total_gt_09": 0, "per_image_max": []}

    # -- patched composite_watermark_on_background ---------------------------
    def patched_composite(bg, wm_aug, opacity, blend_mode, position):
        bg_w, bg_h = bg.size
        wm_w, wm_h = wm_aug.size
        px, py = position
        x1 = max(0, px)
        y1 = max(0, py)
        x2 = min(bg_w, px + wm_w)
        y2 = min(bg_h, py + wm_h)
        if x2 > x1 and y2 > y1:
            wm_x1 = x1 - px
            wm_y1 = y1 - py
            wm_x2 = x2 - px
            wm_y2 = y2 - py
            wm_crop = wm_aug.crop((wm_x1, wm_y1, wm_x2, wm_y2))
            if wm_crop.size[0] > 0 and wm_crop.size[1] > 0:
                wm_a = wm_crop.split()[3]
                # Exactly mirrors generate_dataset.composite_watermark_on_background
                # lines computing wm_a_np -- no RNG involved, deterministic
                # given (wm_aug, opacity, position), which are already fixed
                # by the time this function is called.
                wm_a_np = np.asarray(wm_a, dtype=np.float32) / 255.0 * opacity
                A = state["A"]
                region = A[y1:y2, x1:x2]
                A[y1:y2, x1:x2] = 1.0 - (1.0 - region) * (1.0 - wm_a_np)
        return orig_composite(bg, wm_aug, opacity, blend_mode, position)

    # -- patched apply_post_processing ---------------------------------------
    def patched_post_processing(img):
        # MUST mirror generate_dataset.apply_post_processing LINE BY LINE:
        # same random()/np.random calls, in the same order, with the same
        # short-circuiting. Any deviation here desyncs random.Random's
        # internal state from the reference run and silently corrupts every
        # draw for the rest of the dataset -- the per-image exact-decode
        # checks in this script's report are what prove this mirror is
        # faithful; never trust it without them.
        clean = state["clean"]
        A = state["A"]

        img_np = np.array(img, dtype=np.float32)
        clean_np = np.array(clean, dtype=np.float32)

        # 1. Gaussian noise (60% of the time) -- identical noise array
        # added to both img and clean, then independently clipped (a real
        # source of small clipping-induced nonlinearity right at 0/255,
        # same as JPEG's own inability to represent out-of-range values).
        if random.random() < 0.6:
            noise_sigma = random.uniform(*mod.NOISE_SIGMA_RANGE)
            if noise_sigma > 1:
                noise = np.random.normal(0, noise_sigma, img_np.shape)
                img_np = np.clip(img_np + noise, 0, 255)
                clean_np = np.clip(clean_np + noise, 0, 255)

        # 2. Colour-temperature shift (30% of the time) -- identical shift.
        if random.random() < 0.3:
            shift = random.uniform(-15, 15)
            img_np[:, :, 0] = np.clip(img_np[:, :, 0] + shift, 0, 255)
            img_np[:, :, 2] = np.clip(img_np[:, :, 2] - shift * 0.7, 0, 255)
            clean_np[:, :, 0] = np.clip(clean_np[:, :, 0] + shift, 0, 255)
            clean_np[:, :, 2] = np.clip(clean_np[:, :, 2] - shift * 0.7, 0, 255)

        img_out = Image.fromarray(img_np.astype(np.uint8))
        clean_out = Image.fromarray(clean_np.astype(np.uint8))

        # 3. Down/up resize (20% of the time) -- same factor/filter applied
        # to img, clean AND the alpha accumulator (PIL mode "F", BILINEAR).
        if random.random() < 0.2:
            factor = random.uniform(0.4, 0.8)
            small_size = (max(64, int(img_out.width * factor)), max(64, int(img_out.height * factor)))
            img_out = img_out.resize(small_size, Image.BILINEAR).resize(img_out.size, Image.BILINEAR)
            clean_out = clean_out.resize(small_size, Image.BILINEAR).resize(clean_out.size, Image.BILINEAR)
            a_img = Image.fromarray(np.ascontiguousarray(A, dtype=np.float32), mode="F")
            a_img = a_img.resize(small_size, Image.BILINEAR).resize(mod.TARGET_SIZE, Image.BILINEAR)
            A = np.asarray(a_img, dtype=np.float32)

        state["clean"] = clean_out
        state["A"] = np.clip(A, 0.0, 1.0).astype(np.float32)
        return img_out

    # -- patched generate_single_image ---------------------------------------
    def patched_generate_single_image(idx, bg_path, watermarks, wm_combo):
        if limit is not None and idx >= limit:
            raise _StopReplay()
        clean0 = Image.open(bg_path).convert("RGB").resize(mod.TARGET_SIZE, Image.LANCZOS)
        state["clean"] = clean0
        state["A"] = np.zeros((mod.TARGET_SIZE[1], mod.TARGET_SIZE[0]), dtype=np.float32)
        result = orig_gen_single(idx, bg_path, watermarks, wm_combo)
        captured_results.append({
            "idx": idx,
            "img_filename": result["img_filename"],
            "bg_filename": result["bg_filename"],
            "wm_combo": list(result["wm_combo"]),
            "has_watermark": result["has_watermark"],
            "annotations": result["annotations"],
        })
        return result

    # -- patched save_as_jpeg_with_quality ------------------------------------
    def patched_save_as_jpeg(img, path):
        # Same draw, same position in the RNG sequence as the original --
        # we just redirect what gets written.
        quality = random.randint(*mod.JPEG_QUALITY_RANGE)
        path = Path(path)
        split = path.parent.name
        stem = path.stem

        clean = state["clean"]
        A = np.clip(state["A"], 0.0, 1.0).astype(np.float32)

        clean.save(clean_root / split / f"{stem}.png", "PNG")
        alpha_u8 = np.clip(np.round(A * 255.0), 0, 255).astype(np.uint8)
        Image.fromarray(alpha_u8, mode="L").save(alpha_root / split / f"{stem}.png", "PNG")

        if captured_results:
            captured_results[-1]["split"] = split
            captured_results[-1]["stem"] = stem
            captured_results[-1]["quality"] = int(quality)

        rec = {
            "idx": captured_results[-1]["idx"] if captured_results else None,
            "stem": stem,
            "split": split,
            "quality": int(quality),
            "alpha_max": float(A.max()) if A.size else 0.0,
            "alpha_p50": float(np.percentile(A, 50)),
            "alpha_p99": float(np.percentile(A, 99)),
        }
        pos = A[A > 0]
        rec["alpha_pos_p99"] = float(np.percentile(pos, 99)) if pos.size else 0.0

        stats_acc["per_image_max"].append(rec["alpha_max"])
        stats_acc["total_pixels"] += int(A.size)
        stats_acc["total_pos_pixels"] += int(pos.size)
        stats_acc["total_gt_09"] += int(np.sum(A > 0.9))
        if pos.size:
            h, _ = np.histogram(pos, bins=256, range=(0.0, 1.0))
            hist_counts[:] += h

        n_so_far = len(per_image_records)
        do_verify = (n_so_far % max(1, args.verify_every) == 0)
        if do_verify:
            buf = io.BytesIO()
            img.convert("RGB").save(buf, "JPEG", quality=quality)
            buf.seek(0)
            replay_decoded = np.asarray(Image.open(buf).convert("RGB"), dtype=np.int16)

            ref_path = Path(args.reference_root) / "images" / split / f"{stem}.jpg"
            if ref_path.exists():
                ref_decoded = np.asarray(Image.open(ref_path).convert("RGB"), dtype=np.int16)
                if replay_decoded.shape == ref_decoded.shape:
                    diff = np.abs(replay_decoded - ref_decoded)
                    rec["max_abs_diff"] = int(diff.max())
                    rec["mean_abs_diff"] = float(diff.mean())
                    rec["exact"] = bool(diff.max() == 0)
                else:
                    rec["max_abs_diff"] = None
                    rec["mean_abs_diff"] = None
                    rec["exact"] = False
                    rec["shape_mismatch"] = [list(replay_decoded.shape), list(ref_decoded.shape)]

                pre_jpeg = np.asarray(img.convert("RGB"), dtype=np.float64)
                ref_f = ref_decoded.astype(np.float64)
                if pre_jpeg.shape == ref_f.shape:
                    mse = float(np.mean((pre_jpeg - ref_f) ** 2))
                    rec["psnr_pre_jpeg_vs_ref"] = (float("inf") if mse == 0 else 10.0 * math.log10((255.0 ** 2) / mse))
                rec["verified"] = True
            else:
                rec["verified"] = False
                rec["verify_error"] = f"reference image not found: {ref_path}"
        else:
            rec["verified"] = False

        per_image_records.append(rec)
        return None

    mod.composite_watermark_on_background = patched_composite
    mod.apply_post_processing = patched_post_processing
    mod.generate_single_image = patched_generate_single_image
    mod.save_as_jpeg_with_quality = patched_save_as_jpeg

    old_argv = sys.argv
    sys.argv = [str(gen_path)]
    t0 = time.time()
    try:
        mod.main()
    except _StopReplay:
        pass
    finally:
        sys.argv = old_argv
    elapsed = time.time() - t0

    # ------------------------------------------------------------------
    # Cross-check against reference CSV / COCO attributes. Built from our
    # OWN captured_results rather than gen_out_dir's own annotations/*.json,
    # because when --limit triggers an early _StopReplay, main()'s own
    # post-loop COCO/CSV-writing code (which sits after the per-image for
    # loop) never runs -- captured_results is populated incrementally
    # during the loop instead and works identically with or without
    # --limit.
    # ------------------------------------------------------------------
    csv_mismatches = []
    attr_mismatches = []
    ATTR_KEYS = ("scale", "rotation", "opacity", "blend_mode", "is_tiled")

    ref_csv_path = Path(args.reference_root) / "annotations" / "image_labels.csv"
    ref_rows = {}
    if ref_csv_path.exists():
        with open(ref_csv_path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                ref_rows[row["image_name"]] = row
    else:
        csv_mismatches.append({"reason": f"reference CSV not found: {ref_csv_path}"})

    ref_coco = {}
    for split in ("train", "val"):
        p = Path(args.reference_root) / "annotations" / f"{split}.json"
        if not p.exists():
            attr_mismatches.append({"reason": f"reference COCO file not found: {p}"})
            continue
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        by_img = {img["id"]: [] for img in data["images"]}
        for ann in data["annotations"]:
            by_img.setdefault(ann["image_id"], []).append(ann)
        for anns in by_img.values():
            anns.sort(key=lambda a: a["id"])
        ref_coco[split] = by_img

    for r in captured_results:
        fname = r["img_filename"]
        split = r.get("split")

        ref_row = ref_rows.get(fname)
        if ref_row is None:
            csv_mismatches.append({"image": fname, "reason": "missing in reference CSV"})
        elif ref_row["background_source"] != r["bg_filename"]:
            csv_mismatches.append({
                "image": fname, "reason": "background_source mismatch",
                "reference": ref_row["background_source"], "replay": r["bg_filename"],
            })

        idx = r["idx"]
        ref_anns = ref_coco.get(split, {}).get(idx) if split else None
        if ref_anns is None:
            if split is not None:
                attr_mismatches.append({"image": fname, "reason": f"image id {idx} not found in reference {split}.json"})
            continue
        my_anns = r["annotations"]
        if len(ref_anns) != len(my_anns):
            attr_mismatches.append({
                "image": fname, "reason": "annotation count mismatch",
                "reference_count": len(ref_anns), "replay_count": len(my_anns),
            })
            continue
        for j, (ra, ma) in enumerate(zip(ref_anns, my_anns)):
            rattrs = ra.get("attributes", {})
            mattrs = ma.get("attributes", {})
            for k in ATTR_KEYS:
                rv, mv = rattrs.get(k), mattrs.get(k)
                if rv != mv:
                    attr_mismatches.append({
                        "image": fname, "ann_index": j, "key": k,
                        "reference": rv, "replay": mv,
                    })

    # ------------------------------------------------------------------
    # Alpha distribution stats
    # ------------------------------------------------------------------
    per_image_max = np.asarray(stats_acc["per_image_max"], dtype=np.float64)
    alpha_stats = {
        "per_image_max_p50": float(np.percentile(per_image_max, 50)) if per_image_max.size else 0.0,
        "per_image_max_p99": float(np.percentile(per_image_max, 99)) if per_image_max.size else 0.0,
        "per_image_max_true_max": float(per_image_max.max()) if per_image_max.size else 0.0,
        "positive_pixel_p99": _percentile_from_hist(hist_counts, bin_edges, 99.0),
        "frac_pixels_gt_0.9": (stats_acc["total_gt_09"] / stats_acc["total_pixels"]) if stats_acc["total_pixels"] else 0.0,
        "frac_pixels_positive": (stats_acc["total_pos_pixels"] / stats_acc["total_pixels"]) if stats_acc["total_pixels"] else 0.0,
    }

    # ------------------------------------------------------------------
    # Disk usage
    # ------------------------------------------------------------------
    def _dir_bytes(p: Path) -> int:
        return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

    n_total = len(captured_results)
    disk_bytes = _dir_bytes(clean_root) + _dir_bytes(alpha_root)

    verified = [r for r in per_image_records if r.get("verified")]
    n_exact = sum(1 for r in verified if r.get("exact"))
    max_abs_diffs = [r["max_abs_diff"] for r in verified if r.get("max_abs_diff") is not None]
    mean_abs_diffs = [r["mean_abs_diff"] for r in verified if r.get("mean_abs_diff") is not None]

    n_train = sum(1 for r in captured_results if r.get("split") == "train")
    n_val = sum(1 for r in captured_results if r.get("split") == "val")

    report = {
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "python_version": sys.version,
        "pillow_version": _PIL_VERSION,
        "numpy_version": np.__version__,
        "cv2_version": cv2.__version__,
        "generator_path": str(gen_path),
        "backgrounds_dir": str(args.backgrounds_dir),
        "watermarks_dir": str(args.watermarks_dir),
        "reference_root": str(args.reference_root),
        "limit": limit,
        "verify_every": args.verify_every,
        "elapsed_seconds": elapsed,
        "seconds_per_image": (elapsed / n_total) if n_total else None,
        "disk_bytes_total": disk_bytes,
        "disk_bytes_per_image": (disk_bytes / n_total) if n_total else None,
        "counts": {"total_processed": n_total, "train": n_train, "val": n_val},
        "exact_match": {
            "n_verified": len(verified),
            "n_exact": n_exact,
            "rate": (n_exact / len(verified)) if verified else None,
            "max_abs_diff_overall": max(max_abs_diffs) if max_abs_diffs else None,
            "mean_abs_diff_mean": (sum(mean_abs_diffs) / len(mean_abs_diffs)) if mean_abs_diffs else None,
        },
        "csv_mismatches": {"count": len(csv_mismatches), "examples": csv_mismatches[:10]},
        "attr_mismatches": {"count": len(attr_mismatches), "examples": attr_mismatches[:10]},
        "alpha_stats": alpha_stats,
        "images": per_image_records,
    }

    out_root.mkdir(parents=True, exist_ok=True)
    with open(out_root / "replay_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    _print_summary(report)

    ok = (report["exact_match"]["rate"] in (None, 1.0)) and not csv_mismatches and not attr_mismatches
    report["_ok"] = ok
    return report


def _print_summary(report: dict) -> None:
    print("=" * 70)
    print("  REPLAY REPORT SUMMARY")
    print("=" * 70)
    print(f"  Python: {report['python_version'].splitlines()[0]}")
    print(f"  Pillow {report['pillow_version']}  numpy {report['numpy_version']}  cv2 {report['cv2_version']}")
    c = report["counts"]
    print(f"  Processed: {c['total_processed']} (train={c['train']}, val={c['val']})")
    em = report["exact_match"]
    rate = f"{em['rate']*100:.2f}%" if em["rate"] is not None else "n/a"
    print(f"  Exact-decode match: {em['n_exact']}/{em['n_verified']} ({rate})")
    print(f"  max_abs_diff (over verified): {em['max_abs_diff_overall']}")
    print(f"  mean_abs_diff (avg over verified): {em['mean_abs_diff_mean']}")
    print(f"  CSV mismatches: {report['csv_mismatches']['count']}")
    print(f"  COCO attribute mismatches: {report['attr_mismatches']['count']}")
    a = report["alpha_stats"]
    print(f"  Alpha (per-image max): p50={a['per_image_max_p50']:.4f} p99={a['per_image_max_p99']:.4f} "
          f"max={a['per_image_max_true_max']:.4f}")
    print(f"  Alpha (positive pixels p99): {a['positive_pixel_p99']:.4f}  "
          f"frac>0.9: {a['frac_pixels_gt_0.9']:.6f}")
    print(f"  Elapsed: {report['elapsed_seconds']:.1f}s "
          f"({report['seconds_per_image']:.3f}s/image)" if report['seconds_per_image'] else "")
    if report["disk_bytes_per_image"]:
        print(f"  Disk: {report['disk_bytes_total']/1e6:.1f} MB total, "
              f"{report['disk_bytes_per_image']/1e3:.1f} KB/image")
    print("=" * 70)


def build_arg_parser() -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(
        description="Deterministically replay generate_dataset.py to recover exact alpha/clean targets."
    )
    ap.add_argument("--generator", default=None, help="Path to generate_dataset.py (default: sibling file)")
    ap.add_argument("--backgrounds-dir", required=True)
    ap.add_argument("--watermarks-dir", required=True)
    ap.add_argument("--reference-root", required=True,
                     help="A wm_dataset_out dir (contains images/, annotations/) to verify replay against")
    ap.add_argument("--out", required=True, help="Output targets root (clean/, alpha/, replay_report.json)")
    ap.add_argument("--limit", type=int, default=None, help="Stop after the first N indices")
    ap.add_argument("--verify-every", type=int, default=1, help="Verify every Kth processed image (default 1)")
    ap.add_argument("--overwrite", action="store_true", help="Delete --out first")
    return ap


def main(argv=None):
    args = build_arg_parser().parse_args(argv)
    report = run_replay(args)
    sys.exit(0 if report.get("_ok") else 1)


if __name__ == "__main__":
    main()


### 10.3 Rebuild the training targets (skips if already done)


In [ ]:
import os, sys, subprocess, json, time, pathlib

# Set e.g. 50 for a quick test; None replays all 2000 images (~23 min on a
# Kaggle T4 kernel, ~2.2 GB written under /kaggle/working).
REPLAY_LIMIT = None

def replay_ok(report):
    em = report.get('exact_match', {})
    rate = em.get('rate')
    max_diff = em.get('max_abs_diff_overall')
    csv_mm = report.get('csv_mismatches', {}).get('count', 0)
    attr_mm = report.get('attr_mismatches', {}).get('count', 0)
    return (rate is not None and rate >= 0.99
            and max_diff is not None and max_diff <= 3
            and csv_mm == 0 and attr_mm == 0)

def replay_complete(report):
    counts = report.get('counts', {})
    return report.get('limit') is None and counts.get('total_processed', 0) == 2000

report_path = TARGETS_DIR / 'replay_report.json'
need_replay = True
if report_path.exists():
    report = json.loads(report_path.read_text())
    if replay_complete(report) and replay_ok(report):
        print(f'Found a complete, verified replay report at {report_path} -- skipping replay.')
        need_replay = False
    else:
        print(f'Found an incomplete or non-exact replay report at {report_path} -- re-running.')

if need_replay:
    assert BACKGROUNDS_DIR is not None and WATERMARKS_DIR is not None, (
        'Need wm_backgrounds_v2/ and watermarks_processed/ under /kaggle/input to run the replay -- '
        'see section 10 markdown for what to upload.'
    )

    # Exact replay depends on the image-library build: locally, Pillow 10.4.0
    # + opencv 4.10.0.84 reproduced the reference JPEGs pixel-for-pixel.
    # Kaggle's kernel ships other versions, so install the matching ones into
    # an ISOLATED directory and put it first on the replay subprocess's
    # PYTHONPATH only -- the notebook kernel (torch, ultralytics, training)
    # keeps its own packages. --no-deps keeps Kaggle's numpy; --upgrade avoids
    # "target directory already exists" warnings on a re-run.
    REPLAY_ENV = pathlib.Path('/tmp/replay_env')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--upgrade', '--target', str(REPLAY_ENV),
         'pillow==10.4.0', 'opencv-python-headless==4.10.0.84'],
        check=True,
    )
    replay_env = dict(os.environ)
    replay_env['PYTHONPATH'] = str(REPLAY_ENV) + os.pathsep + replay_env.get('PYTHONPATH', '')

    cmd = [
        sys.executable, '/kaggle/working/replay_alpha_targets.py',
        '--generator', '/kaggle/working/generate_dataset.py',
        '--backgrounds-dir', str(BACKGROUNDS_DIR),
        '--watermarks-dir', str(WATERMARKS_DIR),
        '--reference-root', str(WM_DATASET),
        '--out', str(TARGETS_DIR),
        '--overwrite',
    ]
    if REPLAY_LIMIT is not None:
        cmd += ['--limit', str(REPLAY_LIMIT)]

    print('Running:', ' '.join(cmd))
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=replay_env)
    for line in proc.stdout:
        print(line, end='')
    ret = proc.wait()
    print(f'\nReplay subprocess exited {ret} after {time.time() - t0:.1f}s')

    # The replay script exits 1 whenever it isn't 100.00% pixel-exact (see its
    # own module docstring) -- that alone is not necessarily a failure for us
    # (see replay_ok's tolerance below), so the report on disk, not the exit
    # code, is what decides whether to proceed.
    if not report_path.exists():
        raise RuntimeError(
            f'The replay exited with code {ret} without writing {report_path} -- it crashed before '
            'finishing (see the log above, e.g. a missing background or watermark file).'
        )
    report = json.loads(report_path.read_text())

em = report['exact_match']
print('\n=== replay report summary ===')
print(f"processed: {report['counts']['total_processed']} (train={report['counts']['train']}, val={report['counts']['val']})")
print(f"exact-decode match: {em['n_exact']}/{em['n_verified']} (rate={em['rate']})")
print(f"max_abs_diff_overall: {em['max_abs_diff_overall']}")
print(f"CSV mismatches: {report['csv_mismatches']['count']}, COCO attribute mismatches: {report['attr_mismatches']['count']}")
print('alpha_stats:', report['alpha_stats'])

if not replay_ok(report):
    raise RuntimeError(
        'Replay did not reproduce the reference dataset closely enough to trust as training supervision -- '
        'this looks like a real desync (mismatched generator/library parameters, or large pixel differences) '
        'rather than ordinary JPEG-decode rounding. Check the replay log above and '
        'scripts/kaggle_dataset/README.md.'
    )

if report.get('limit') is not None:
    print(f"\nNOTE: this replay report has limit={report['limit']} -- targets are PARTIAL "
          f"({report['counts']['total_processed']} of 2000 images). Training below will only use those images.")


### 10.4 Check the targets


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

pos_sample = None  # (split, stem) of the first sample found with a real mark

for split in ['train', 'val']:
    img_dir = WM_DATASET / 'images' / split
    alpha_dir = TARGETS_DIR / 'alpha' / split
    clean_dir = TARGETS_DIR / 'clean' / split
    img_stems = {p.stem for p in img_dir.glob('*.jpg')}
    alpha_stems = {p.stem for p in alpha_dir.glob('*.png')} if alpha_dir.is_dir() else set()
    clean_stems = {p.stem for p in clean_dir.glob('*.png')} if clean_dir.is_dir() else set()
    complete = img_stems & alpha_stems & clean_stems
    print(f'{split:5} images={len(img_stems):4d} alpha={len(alpha_stems):4d} clean={len(clean_stems):4d} '
          f'complete={len(complete):4d}')
    assert complete, f'{split}: no stem has both a replayed alpha and clean target -- run 10.3 first.'
    if pos_sample is None:
        for stem in sorted(complete):
            a = cv2.imread(str(alpha_dir / f'{stem}.png'), cv2.IMREAD_GRAYSCALE)
            if a is not None and a.max() / 255.0 >= 0.1:
                pos_sample = (split, stem)
                break

assert pos_sample is not None, 'No positive (visibly watermarked) sample found across train/val to sanity-check.'
split, stem = pos_sample

obs_path = next((WM_DATASET / 'images' / split).glob(f'{stem}.*'))
obs = cv2.cvtColor(cv2.imread(str(obs_path)), cv2.COLOR_BGR2RGB).astype(np.float32)
clean = cv2.cvtColor(cv2.imread(str(TARGETS_DIR / 'clean' / split / f'{stem}.png')), cv2.COLOR_BGR2RGB).astype(np.float32)
alpha = cv2.imread(str(TARGETS_DIR / 'alpha' / split / f'{stem}.png'), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

diff = np.abs(obs - clean).mean(axis=-1)
inside = alpha >= 0.08
outside = alpha < 1e-6
print(f'\nSample {split}/{stem}: alpha max = {alpha.max():.3f}')
print(f'  mean |obs-clean| outside mark (JPEG floor): {diff[outside].mean():.3f}')
print(f'  mean |obs-clean| inside mark (signal)      : {diff[inside].mean():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(np.clip(obs / 255.0, 0, 1)); axes[0].set_title('observed')
axes[1].imshow(np.clip(clean / 255.0, 0, 1)); axes[1].set_title('clean (replayed)')
axes[2].imshow(alpha, cmap='inferno', vmin=0, vmax=1); axes[2].set_title('alpha (replayed)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/alpha_targets_preview.png', dpi=120)
plt.show()


### 10.5 Model

`AlphaUNet` inline, copied from `watermark_remover/alpha_net.py` (see that module's docstring for the full physics/architecture writeup and the calibration behind `ALPHA_MAX = 0.9`). Kept identical (module names, parameter shapes, forward math) so a checkpoint trained here loads there via `alpha_net.load_model`, and vice versa.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

ALPHA_MAX = 0.9                    # calibrated for this dataset -- see watermark_remover/alpha_net.py docstring
WIDTHS = (32, 64, 128, 256)
FORMAT_VERSION = 1                 # bump together with watermark_remover/alpha_net.py if the checkpoint format changes
ALPHA_BIAS_INIT = -4.0             # sigmoid(-4) ~= 0.018 -> alpha_max*sigmoid(-4) ~= 0.016 at init (near-identity)

class ConvBlock(nn.Module):
    """Two x (Conv3x3 -> GroupNorm(8) -> SiLU), same spatial size in/out."""

    def __init__(self, in_ch, out_ch, dilation=1):
        super().__init__()
        padding = dilation
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=padding, dilation=dilation)
        self.gn1 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=padding, dilation=dilation)
        self.gn2 = nn.GroupNorm(8, out_ch)
        self.act = nn.SiLU(inplace=True)

    def forward(self, x):
        x = self.act(self.gn1(self.conv1(x)))
        x = self.act(self.gn2(self.conv2(x)))
        return x

class AlphaUNet(nn.Module):
    """Small U-Net predicting per-pixel watermark opacity (alpha) and ink
    colour from an RGB crop (N,3,H,W in [0,1], H/W multiples of 16). Encoder
    of len(widths) ConvBlocks with MaxPool2d(2) between levels; a dilated
    bottleneck at widths[-1]; a decoder mirroring the encoder (bilinear
    upsample -> concat skip -> ConvBlock); a 1x1 head -> 4 channels (alpha via
    alpha_max*sigmoid, ink RGB via sigmoid). Must stay identical (attribute
    names, parameter shapes, forward math) to watermark_remover/alpha_net.py's
    AlphaUNet -- state_dicts are interchangeable between the two."""

    def __init__(self, widths=WIDTHS, alpha_max=ALPHA_MAX):
        super().__init__()
        widths = tuple(widths)
        assert len(widths) >= 2, f'AlphaUNet needs at least 2 widths, got {widths!r}'
        self.widths = widths
        self.alpha_max = float(alpha_max)
        self.pool = nn.MaxPool2d(2)

        enc_in = (3,) + widths[:-1]
        self.encoders = nn.ModuleList([ConvBlock(c_in, c_out) for c_in, c_out in zip(enc_in, widths)])
        self.bottleneck = ConvBlock(widths[-1], widths[-1], dilation=2)

        dec_in_ch = [widths[i + 1] + widths[i] for i in range(len(widths) - 1)]
        dec_out_ch = list(widths[:-1])
        self.decoders = nn.ModuleList([ConvBlock(c_in, c_out) for c_in, c_out in zip(dec_in_ch, dec_out_ch)])

        self.head = nn.Conv2d(widths[0], 4, kernel_size=1)
        with torch.no_grad():
            self.head.bias.zero_()
            self.head.bias[0] = ALPHA_BIAS_INIT

    def forward(self, x):
        skips = []
        h = x
        n = len(self.encoders)
        for i, enc in enumerate(self.encoders):
            h = enc(h)
            skips.append(h)
            if i < n - 1:
                h = self.pool(h)
        h = self.bottleneck(h)
        for dec, skip in zip(self.decoders[::-1], skips[-2::-1]):
            h = F.interpolate(h, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            h = torch.cat([h, skip], dim=1)
            h = dec(h)
        out = self.head(h)
        alpha = self.alpha_max * torch.sigmoid(out[:, 0:1])
        ink = torch.sigmoid(out[:, 1:4])
        return alpha, ink

def recover(obs, alpha, ink, alpha_max):
    """Closed-form inverse of obs = alpha*ink + (1-alpha)*true, solved for
    true. Denominator floored at 1 - alpha_max so a worst-case prediction can
    never blow up the division beyond what alpha_max itself bounds."""
    denom = torch.clamp(1.0 - alpha, min=1.0 - alpha_max)
    return torch.clamp((obs - alpha * ink) / denom, 0.0, 1.0)

def save_checkpoint(path, model, epoch, val_metrics, config):
    """Writes exactly the repo's alpha_net.checkpoint_dict format -- only
    tensors/plain Python primitives, so both torch.load(weights_only=True)
    and watermark_remover.alpha_net.load_model can read the result. Optimizer
    /scheduler/scaler state (needed only to resume, not to just USE the
    weights) is saved separately, in last.pt."""
    ckpt = {
        'format_version': FORMAT_VERSION,
        'arch': 'AlphaUNet',
        'widths': list(model.widths),
        'alpha_max': float(model.alpha_max),
        'model_state': model.state_dict(),
        'epoch': epoch,
        'val_metrics': val_metrics,
        'train_config': config,
    }
    torch.save(ckpt, path)

_probe_model = AlphaUNet()
_n_params = sum(p.numel() for p in _probe_model.parameters())
print(f'AlphaUNet: widths={WIDTHS} alpha_max={ALPHA_MAX} params={_n_params:,}')
del _probe_model


### 10.6 Dataset and loss


In [ ]:
import cv2
import numpy as np
import torch

IMG_EXTS = ('.jpg', '.jpeg', '.png')

def dilate_bool_mask(mask_bool, ksize):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    return cv2.dilate(mask_bool.astype(np.uint8), kernel).astype(np.float32)

class AlphaCropDataset(torch.utils.data.Dataset):
    """Random (train) or deterministic (val) crops pairing
    WM_DATASET/images/<split> with TARGETS_DIR/{alpha,clean}/<split> by
    filename stem. Returns per item: obs (3,crop,crop), clean (3,crop,crop),
    alpha (1,crop,crop, clamped to ALPHA_MAX), w_alpha (1,crop,crop), w_rec
    (1,crop,crop) -- float32 tensors, obs/clean in [0, 1].

    w_alpha and w_rec both start from weight = 1 + 4*dilated(alpha > 0.01)
    (mark pixels, and a halo around them, count more), then diverge:
    w_alpha also requires either alpha<=0.01 (true background, still
    supervised) or the pixel is actually VISIBLE after gain jitter (a pixel
    with alpha>0 that is not visibly different from clean, e.g. "multiply"
    onto a near-white patch, carries no recoverable alpha signal). w_rec
    excludes alpha > ALPHA_MAX entirely -- recover's denominator is floored
    at 1 - ALPHA_MAX, so those pixels are architecturally unrecoverable and
    penalising the network for that ceiling would teach the wrong thing.
    """

    def __init__(self, split, crop, samples_per_epoch=None, deterministic=False, seed=0):
        self.split = split
        self.crop = int(crop)
        self.samples_per_epoch = samples_per_epoch
        self.deterministic = deterministic
        self.seed = seed

        img_dir = WM_DATASET / 'images' / split
        alpha_dir = TARGETS_DIR / 'alpha' / split
        clean_dir = TARGETS_DIR / 'clean' / split
        imgs = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS}
        alphas = {p.stem: p for p in alpha_dir.glob('*.png')} if alpha_dir.is_dir() else {}
        cleans = {p.stem: p for p in clean_dir.glob('*.png')} if clean_dir.is_dir() else {}
        stems = sorted(set(imgs) & set(alphas) & set(cleans))
        n_skipped = len(set(imgs) - set(stems))
        print(f'[AlphaCropDataset:{split}] {len(stems)} usable stem(s), {n_skipped} skipped (no replayed target)')
        assert stems, f'No stems with both a replayed alpha and clean target for split={split!r} -- run 10.3 first.'
        self.items = [(imgs[s], alphas[s], cleans[s]) for s in stems]

    def __len__(self):
        if self.samples_per_epoch is not None:
            return int(self.samples_per_epoch)
        if self.deterministic:
            return 2 * len(self.items)
        return len(self.items)

    def _read_triple(self, i):
        img_p, alpha_p, clean_p = self.items[i]
        obs = cv2.cvtColor(cv2.imread(str(img_p), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        clean = cv2.cvtColor(cv2.imread(str(clean_p), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        alpha = cv2.imread(str(alpha_p), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        return obs, clean, alpha

    def __getitem__(self, idx):
        n = len(self.items)
        if self.deterministic:
            # Fixed function of (seed, idx) -- independent of worker count or call order.
            rng = np.random.default_rng([int(self.seed), int(idx)])
            img_idx = (idx // 2) % n
            prefer_wm = (idx % 2 == 0)
        else:
            rng = np.random.default_rng()
            img_idx = int(rng.integers(0, n))
            prefer_wm = rng.random() < 0.7

        obs, clean, alpha = self._read_triple(img_idx)
        crop = self.crop
        h, w = alpha.shape[:2]
        pad_h, pad_w = max(0, crop - h), max(0, crop - w)
        if pad_h or pad_w:
            obs = np.pad(obs, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
            clean = np.pad(clean, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
            alpha = np.pad(alpha, ((0, pad_h), (0, pad_w)), mode='constant', constant_values=0.0)

        h, w = alpha.shape[:2]
        max_y0, max_x0 = max(0, h - crop), max(0, w - crop)
        has_wm_px = bool(np.any(alpha > 0.05))
        if prefer_wm and has_wm_px:
            ys, xs = np.nonzero(alpha > 0.05)
            i = int(rng.integers(0, len(ys)))
            y0 = int(np.clip(ys[i] - crop // 2, 0, max_y0))
            x0 = int(np.clip(xs[i] - crop // 2, 0, max_x0))
        else:
            y0 = int(rng.integers(0, max_y0 + 1))
            x0 = int(rng.integers(0, max_x0 + 1))

        obs_c = obs[y0:y0 + crop, x0:x0 + crop, :].astype(np.float32).copy()
        clean_c = clean[y0:y0 + crop, x0:x0 + crop, :].astype(np.float32).copy()
        alpha_c = alpha[y0:y0 + crop, x0:x0 + crop].astype(np.float32).copy()

        gain = rng.uniform(0.85, 1.0, size=(1, 1, 3)).astype(np.float32)
        obs_c = obs_c * gain
        clean_c = clean_c * gain

        weight_c = 1.0 + 4.0 * dilate_bool_mask(alpha_c > 0.01, 7)
        diff_mean = np.abs(obs_c - clean_c).mean(axis=2)
        visible = dilate_bool_mask(diff_mean > (4.0 / 255.0), 3)
        w_alpha_c = weight_c * np.maximum((alpha_c <= 0.01).astype(np.float32), visible)
        w_rec_c = weight_c * (alpha_c <= ALPHA_MAX).astype(np.float32)
        alpha_c = np.clip(alpha_c, 0.0, ALPHA_MAX)

        obs_t = torch.from_numpy(np.ascontiguousarray(obs_c.transpose(2, 0, 1)))
        clean_t = torch.from_numpy(np.ascontiguousarray(clean_c.transpose(2, 0, 1)))
        alpha_t = torch.from_numpy(np.ascontiguousarray(alpha_c[None, :, :]))
        w_alpha_t = torch.from_numpy(np.ascontiguousarray(w_alpha_c[None, :, :]))
        w_rec_t = torch.from_numpy(np.ascontiguousarray(w_rec_c[None, :, :]))
        return obs_t, clean_t, alpha_t, w_alpha_t, w_rec_t


def alpha_loss(pred_alpha, pred_ink, obs, clean, alpha, w_alpha, w_rec):
    """5*L_alpha + 1*L_rec + 0.5*L_comp -- alpha target is already clamped to
    ALPHA_MAX by the dataset. Returns (total, {'L_alpha':..., 'L_rec':...,
    'L_comp':...}) with the parts already .item()'d."""
    rec = recover(obs, pred_alpha, pred_ink, ALPHA_MAX)
    l_alpha = (w_alpha * (pred_alpha - alpha).abs()).mean()
    l_rec = (w_rec * (rec - clean).abs().mean(dim=1, keepdim=True)).mean()
    comp = pred_alpha * pred_ink + (1.0 - pred_alpha) * clean
    l_comp = (w_rec * (comp - obs).abs().mean(dim=1, keepdim=True)).mean()
    total = 5.0 * l_alpha + 1.0 * l_rec + 0.5 * l_comp
    parts = {'L_alpha': float(l_alpha.item()), 'L_rec': float(l_rec.item()), 'L_comp': float(l_comp.item())}
    return total, parts


def evaluate(model, loader, device):
    """Pixel-accurate validation metrics accumulated over the whole loader,
    split into watermark (alpha_gt > 0.01) and background regions; pixels
    with alpha_gt > ALPHA_MAX are excluded from every metric (architecturally
    unrecoverable -- frac_alpha_gt_max reports how much of the set that was).
    val_score is lower-is-better and penalises recover damaging clean pixels.
    """
    model.eval()
    sums = dict(alpha_err_wm=0.0, n_wm=0, alpha_bg=0.0, n_bg=0,
                rec_err_wm=0.0, id_err_wm=0.0, rec_err_bg=0.0, id_err_bg=0.0,
                n_excluded=0, n_total=0)
    with torch.no_grad():
        for obs, clean, alpha, _w_alpha, _w_rec in loader:
            obs, clean, alpha = obs.to(device), clean.to(device), alpha.to(device)
            pred_alpha, pred_ink = model(obs)
            rec = recover(obs, pred_alpha, pred_ink, ALPHA_MAX)

            recoverable = alpha <= ALPHA_MAX
            wm_mask = (alpha > 0.01) & recoverable
            bg_mask = (~(alpha > 0.01)) & recoverable

            alpha_err = (pred_alpha - alpha).abs()
            rec_err = (rec - clean).abs().mean(dim=1, keepdim=True) * 255.0
            id_err = (obs - clean).abs().mean(dim=1, keepdim=True) * 255.0

            n_wm, n_bg = int(wm_mask.sum().item()), int(bg_mask.sum().item())
            if n_wm:
                sums['alpha_err_wm'] += float(alpha_err[wm_mask].sum().item())
                sums['rec_err_wm'] += float(rec_err[wm_mask].sum().item())
                sums['id_err_wm'] += float(id_err[wm_mask].sum().item())
                sums['n_wm'] += n_wm
            if n_bg:
                sums['alpha_bg'] += float(pred_alpha[bg_mask].sum().item())
                sums['rec_err_bg'] += float(rec_err[bg_mask].sum().item())
                sums['id_err_bg'] += float(id_err[bg_mask].sum().item())
                sums['n_bg'] += n_bg
            sums['n_excluded'] += int((~recoverable).sum().item())
            sums['n_total'] += int(alpha.numel())

    n_wm, n_bg = max(sums['n_wm'], 1), max(sums['n_bg'], 1)
    alpha_mae_wm = sums['alpha_err_wm'] / n_wm
    alpha_mean_bg = sums['alpha_bg'] / n_bg
    rec_mae_wm_255 = sums['rec_err_wm'] / n_wm
    identity_mae_wm_255 = sums['id_err_wm'] / n_wm
    rec_mae_bg_255 = sums['rec_err_bg'] / n_bg
    identity_mae_bg_255 = sums['id_err_bg'] / n_bg
    frac_alpha_gt_max = sums['n_excluded'] / max(sums['n_total'], 1)
    improvement_wm = (1.0 - rec_mae_wm_255 / identity_mae_wm_255) if identity_mae_wm_255 > 1e-8 else 0.0
    val_score = rec_mae_wm_255 + 2.0 * max(0.0, rec_mae_bg_255 - identity_mae_bg_255)

    return {
        'alpha_mae_wm': alpha_mae_wm, 'alpha_mean_bg': alpha_mean_bg,
        'rec_mae_wm_255': rec_mae_wm_255, 'identity_mae_wm_255': identity_mae_wm_255,
        'improvement_wm': improvement_wm, 'rec_mae_bg_255': rec_mae_bg_255,
        'identity_mae_bg_255': identity_mae_bg_255, 'frac_alpha_gt_max': frac_alpha_gt_max,
        'val_score': val_score,
    }


PROBE_CROP = 512       # only for this cell's sanity-check batch -- 10.7 rebuilds the datasets with its own constants
PROBE_SAMPLES = 8
train_ds = AlphaCropDataset('train', PROBE_CROP, samples_per_epoch=PROBE_SAMPLES, deterministic=False, seed=0)
val_ds = AlphaCropDataset('val', PROBE_CROP, samples_per_epoch=None, deterministic=True, seed=0)
print(f'train_ds: {len(train_ds)} samples/epoch, {len(train_ds.items)} source image(s)')
print(f'val_ds:   {len(val_ds)} crops, {len(val_ds.items)} source image(s)')

probe_loader = torch.utils.data.DataLoader(train_ds, batch_size=2, shuffle=False, num_workers=0)
obs_b, clean_b, alpha_b, w_alpha_b, w_rec_b = next(iter(probe_loader))
print('batch shapes:', obs_b.shape, clean_b.shape, alpha_b.shape, w_alpha_b.shape, w_rec_b.shape)


### 10.7 Train

This is the actual training cell. Expected wall-clock time is **unknown until measured** -- read the first epoch's `time=...s` in the printed log and scale by `EPOCHS` to estimate the whole run. A partial run is still useful: `best.pt`/`last.pt` are updated every epoch, and re-running this cell resumes from `last.pt` where it left off.

`ALPHA_MAX = 0.9` (10.5) is calibrated for THIS dataset's per-mark opacity range stacked up to 3 layers -- see `watermark_remover/alpha_net.py`'s module docstring for the measured alpha distribution and why the ceiling matters (recovery gets noisier as predicted alpha approaches it).

If you hit a CUDA out-of-memory error: first drop `BATCH_SIZE` from 8 to 4, then (if still OOM) drop `CROP` from 512 to 384.


In [ ]:
import csv, math, time, shutil, pathlib
import torch
from IPython.display import FileLink, display

EPOCHS = 40
BATCH_SIZE = 8
CROP = 512
SAMPLES_PER_EPOCH = 3000
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
PATIENCE = 12                 # early-stop after this many epochs with no val_score improvement
SEED = 0
SNAPSHOT_EVERY = 10           # copy best.pt -> alpha_net_best_epoch_N.pt every N epochs

torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

ALPHA_RUN_DIR.mkdir(parents=True, exist_ok=True)
LAST_PATH = ALPHA_RUN_DIR / 'last.pt'
BEST_PATH = ALPHA_RUN_DIR / 'best.pt'
METRICS_PATH = ALPHA_RUN_DIR / 'metrics.csv'

train_ds = AlphaCropDataset('train', CROP, samples_per_epoch=SAMPLES_PER_EPOCH, deterministic=False, seed=SEED)
val_ds = AlphaCropDataset('val', CROP, samples_per_epoch=None, deterministic=True, seed=SEED)
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(device == 'cuda'), drop_last=True, persistent_workers=(NUM_WORKERS > 0),
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device == 'cuda'), persistent_workers=(NUM_WORKERS > 0),
)
iters_per_epoch = len(train_loader)
assert iters_per_epoch > 0, 'Training loader produced zero batches -- reduce BATCH_SIZE or raise SAMPLES_PER_EPOCH.'

model = AlphaUNet(widths=WIDTHS, alpha_max=ALPHA_MAX).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

warmup_iters = iters_per_epoch  # 1-epoch linear warmup
total_iters = max(1, EPOCHS * iters_per_epoch)

def lr_lambda(it):
    if it < warmup_iters:
        return (it + 1) / max(1, warmup_iters)
    progress = min(max((it - warmup_iters) / max(1, total_iters - warmup_iters), 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
use_amp = (device == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

start_epoch = 1
best_score = float('inf')
epochs_no_improve = 0

if LAST_PATH.exists():
    print('Continuing training from checkpoint:', LAST_PATH)
    # weights_only=False: last.pt is written exclusively by this same cell and
    # carries optimizer/scheduler/scaler state (not just tensors/primitives),
    # so it cannot use the stricter weights_only=True loading best.pt uses.
    ckpt = torch.load(LAST_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    opt.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    best_score = ckpt.get('best_score', float('inf'))
    epochs_no_improve = ckpt.get('epochs_no_improve', 0)
    start_epoch = int(ckpt['epoch']) + 1
else:
    print('Starting fresh training')

TRAIN_CONFIG = dict(
    epochs=EPOCHS, batch_size=BATCH_SIZE, crop=CROP, samples_per_epoch=SAMPLES_PER_EPOCH,
    lr=LR, weight_decay=WEIGHT_DECAY, num_workers=NUM_WORKERS, patience=PATIENCE, seed=SEED,
    widths=list(WIDTHS), alpha_max=ALPHA_MAX,
)

best_epoch = None
best_metrics = None

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t_epoch0 = time.time()
    part_sums = {'total': 0.0, 'L_alpha': 0.0, 'L_rec': 0.0, 'L_comp': 0.0}
    n_batches = 0

    for obs, clean, alpha, w_alpha, w_rec in train_loader:
        obs = obs.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)
        alpha = alpha.to(device, non_blocking=True)
        w_alpha = w_alpha.to(device, non_blocking=True)
        w_rec = w_rec.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)
        try:
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                pred_alpha, pred_ink = model(obs)
                loss, parts = alpha_loss(pred_alpha, pred_ink, obs, clean, alpha, w_alpha, w_rec)
            if use_amp:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                raise RuntimeError(
                    f'CUDA out of memory during a training step (batch_size={BATCH_SIZE}, crop={CROP}). '
                    'Try a smaller BATCH_SIZE (e.g. 4) and/or CROP (e.g. 384).'
                ) from e
            raise

        scheduler.step()
        part_sums['total'] += float(loss.item())
        for k in ('L_alpha', 'L_rec', 'L_comp'):
            part_sums[k] += parts[k]
        n_batches += 1

    n_batches = max(1, n_batches)
    train_parts = {k: v / n_batches for k, v in part_sums.items()}
    val_metrics = evaluate(model, val_loader, device)
    epoch_seconds = time.time() - t_epoch0
    lr_now = scheduler.get_last_lr()[0]

    print(
        f'epoch {epoch}/{EPOCHS} train_loss={train_parts["total"]:.4f} '
        f'(alpha={train_parts["L_alpha"]:.4f} rec={train_parts["L_rec"]:.4f} comp={train_parts["L_comp"]:.4f}) '
        f'val_score={val_metrics["val_score"]:.4f} alpha_mae_wm={val_metrics["alpha_mae_wm"]:.4f} '
        f'rec_mae_wm_255={val_metrics["rec_mae_wm_255"]:.2f} lr={lr_now:.2e} time={epoch_seconds:.1f}s'
    )

    is_new_file = not METRICS_PATH.exists()
    fieldnames = (['epoch', 'train_loss', 'train_L_alpha', 'train_L_rec', 'train_L_comp', 'lr', 'seconds']
                  + list(val_metrics.keys()))
    with open(METRICS_PATH, 'a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if is_new_file:
            w.writeheader()
        row = {'epoch': epoch, 'train_loss': train_parts['total'], 'train_L_alpha': train_parts['L_alpha'],
               'train_L_rec': train_parts['L_rec'], 'train_L_comp': train_parts['L_comp'],
               'lr': lr_now, 'seconds': epoch_seconds}
        row.update(val_metrics)
        w.writerow(row)

    improved = val_metrics['val_score'] < best_score
    if improved:
        best_score = val_metrics['val_score']
        epochs_no_improve = 0
        save_checkpoint(BEST_PATH, model, epoch, val_metrics, TRAIN_CONFIG)
        best_epoch, best_metrics = epoch, val_metrics
    else:
        epochs_no_improve += 1

    last_ckpt = {
        'format_version': FORMAT_VERSION, 'arch': 'AlphaUNet', 'widths': list(model.widths),
        'alpha_max': float(model.alpha_max), 'model_state': model.state_dict(),
        'epoch': epoch, 'val_metrics': val_metrics, 'train_config': TRAIN_CONFIG,
        'optimizer': opt.state_dict(), 'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
        'best_score': best_score, 'epochs_no_improve': epochs_no_improve,
    }
    torch.save(last_ckpt, LAST_PATH)

    if epoch % SNAPSHOT_EVERY == 0 and BEST_PATH.exists():
        snap_path = pathlib.Path(f'/kaggle/working/alpha_net_best_epoch_{epoch}.pt')
        shutil.copy2(BEST_PATH, snap_path)
        print(f'\n[Checkpoint] Saved best weights at epoch {epoch} to {snap_path}')
        display(FileLink(snap_path.name))

    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch} ({epochs_no_improve} epochs without improvement)')
        break

print('\n=== training summary ===')
print('best epoch:', best_epoch)
print('best val metrics:', best_metrics)


### 10.8 Validation examples


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

BEST_PATH = ALPHA_RUN_DIR / 'best.pt'
assert BEST_PATH.exists(), f'No trained weights at {BEST_PATH} -- run the training cell (10.7) first.'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(BEST_PATH, map_location=device, weights_only=True)
val_model = AlphaUNet(widths=tuple(ckpt['widths']), alpha_max=float(ckpt['alpha_max'])).to(device)
val_model.load_state_dict(ckpt['model_state'])
val_model.eval()

val_ds = AlphaCropDataset('val', CROP, samples_per_epoch=None, deterministic=True, seed=SEED)

N_SHOW = 4
fig, axes = plt.subplots(N_SHOW, 4, figsize=(16, 4 * N_SHOW))
for row in range(N_SHOW):
    idx = row % len(val_ds)
    obs, clean, alpha_gt, _w_alpha, _w_rec = val_ds[idx]
    with torch.no_grad():
        pred_alpha, pred_ink = val_model(obs.unsqueeze(0).to(device))
        rec = recover(obs.unsqueeze(0).to(device), pred_alpha, pred_ink, float(ckpt['alpha_max']))

    obs_img = obs.permute(1, 2, 0).numpy()
    clean_img = clean.permute(1, 2, 0).numpy()
    alpha_img = pred_alpha[0, 0].cpu().numpy()
    rec_img = rec[0].permute(1, 2, 0).cpu().numpy()

    axes[row, 0].imshow(np.clip(obs_img, 0, 1)); axes[row, 0].set_title('observed')
    axes[row, 1].imshow(alpha_img, cmap='inferno', vmin=0, vmax=float(ckpt['alpha_max'])); axes[row, 1].set_title('predicted alpha')
    axes[row, 2].imshow(np.clip(rec_img, 0, 1)); axes[row, 2].set_title('recovered')
    axes[row, 3].imshow(np.clip(clean_img, 0, 1)); axes[row, 3].set_title('clean')
    for c in range(4):
        axes[row, c].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/alpha_net_val_examples.png', dpi=120)
plt.show()

print('best epoch:', ckpt['epoch'])
print('best val metrics:', ckpt['val_metrics'])


### 10.9 Real documents (images_scraped)

These panels show whether the synthetic training transfers to real scanned/scraped pages -- there is no ground truth here, so this is descriptive only, not an accuracy measurement. A genuinely clean real page should get alpha close to 0 everywhere; if it doesn't, that is a sign of synthetic-to-real gap, not necessarily a code bug.


In [ ]:
import pathlib, time
import numpy as np
import torch
from PIL import Image

def feather_weight(tile, overlap):
    overlap = max(1, min(overlap, tile // 2))
    ramp = np.linspace(1.0 / (overlap + 1), 1.0, overlap, dtype=np.float32)
    w = np.ones(tile, dtype=np.float32)
    w[:overlap] = ramp
    w[-overlap:] = ramp[::-1]
    return np.outer(w, w)

def predict_page(model, img_rgb_uint8, device, tile=512, overlap=64):
    """Tiled full-page inference: reflect-pads to at least one tile and a
    multiple of 16, runs the model over overlapping tiles, blends alpha/ink
    with a feathered weight so tile seams don't show, then calls `recover`
    ONCE on the merged maps (not per-tile) -- same behaviour as
    watermark_remover/alpha_net.py's predict_image."""
    alpha_max = model.alpha_max
    h, w = img_rgb_uint8.shape[:2]
    pad_h = max(tile, int(np.ceil(h / 16.0) * 16))
    pad_w = max(tile, int(np.ceil(w / 16.0) * 16))
    img_f = img_rgb_uint8.astype(np.float32) / 255.0
    img_padded = np.pad(img_f, ((0, pad_h - h), (0, pad_w - w), (0, 0)), mode='reflect')

    alpha_acc = np.zeros((pad_h, pad_w), dtype=np.float32)
    ink_acc = np.zeros((pad_h, pad_w, 3), dtype=np.float32)
    weight_acc = np.zeros((pad_h, pad_w), dtype=np.float32)

    stride = max(1, tile - overlap)
    ys = list(range(0, max(1, pad_h - tile + 1), stride))
    if ys[-1] != pad_h - tile:
        ys.append(pad_h - tile)
    xs = list(range(0, max(1, pad_w - tile + 1), stride))
    if xs[-1] != pad_w - tile:
        xs.append(pad_w - tile)
    ramp2d = feather_weight(tile, overlap)

    with torch.no_grad():
        for y in ys:
            for x in xs:
                crop = img_padded[y:y + tile, x:x + tile, :]
                t = torch.from_numpy(crop).permute(2, 0, 1).unsqueeze(0).to(device)
                a, ink = model(t)
                a_np = a[0, 0].float().cpu().numpy()
                ink_np = ink[0].float().permute(1, 2, 0).cpu().numpy()
                alpha_acc[y:y + tile, x:x + tile] += a_np * ramp2d
                ink_acc[y:y + tile, x:x + tile, :] += ink_np * ramp2d[..., None]
                weight_acc[y:y + tile, x:x + tile] += ramp2d

    weight_safe = np.clip(weight_acc, 1e-6, None)
    alpha_full = (alpha_acc / weight_safe)[:h, :w]
    ink_full = (ink_acc / weight_safe[..., None])[:h, :w, :]

    obs_t = torch.from_numpy(img_f).permute(2, 0, 1).unsqueeze(0)
    alpha_t = torch.from_numpy(alpha_full).unsqueeze(0).unsqueeze(0)
    ink_t = torch.from_numpy(ink_full).permute(2, 0, 1).unsqueeze(0)
    recovered_t = recover(obs_t, alpha_t, ink_t, alpha_max)
    recovered = np.clip(np.round(recovered_t[0].permute(1, 2, 0).numpy() * 255.0), 0, 255).astype(np.uint8)
    return alpha_full.astype(np.float32), ink_full.astype(np.float32), recovered

assert SCRAPED_DIR is not None, 'Could not find images_scraped -- see 10.1.'
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
real_images = sorted(p for p in SCRAPED_DIR.rglob('*') if p.suffix.lower() in EXTS and p.is_file())
assert real_images, f'No images found under {SCRAPED_DIR}'

OUT_DIR = pathlib.Path('/kaggle/working/alpha_net_real')
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(ALPHA_RUN_DIR / 'best.pt', map_location=device, weights_only=True)
real_model = AlphaUNet(widths=tuple(ckpt['widths']), alpha_max=float(ckpt['alpha_max'])).to(device)
real_model.load_state_dict(ckpt['model_state'])
real_model.eval()

panel_paths = []
for p in real_images[:6]:
    img = np.array(Image.open(p).convert('RGB'))
    t0 = time.time()
    alpha, ink, recovered = predict_page(real_model, img, device)
    dt = time.time() - t0

    alpha_vis = (np.clip(alpha / max(float(ckpt['alpha_max']), 1e-6), 0, 1) * 255).astype(np.uint8)
    alpha_vis = np.stack([alpha_vis] * 3, axis=-1)
    panel = np.concatenate([img, alpha_vis, recovered], axis=1)
    panel_path = OUT_DIR / f'{p.stem}_alpha_panel.png'
    Image.fromarray(panel).save(panel_path)
    panel_paths.append(panel_path)

    frac_marked = float((alpha > 0.05).mean()) * 100
    print(f'{p.name}: alpha_max={alpha.max():.3f} marked={frac_marked:.2f}% time={dt:.2f}s')

from IPython.display import Image as IPyImage, display
for path in panel_paths[:3]:
    display(IPyImage(filename=str(path), width=1200))


### 10.10 Download the final model

Copies the best checkpoint to a single, stably-named file and shows its epoch/metrics/size. To wire this into the app, place the downloaded file at **`weights/alpha_net_best_final.pt`** in the repo -- that enables Method 4's **"Alpha Network (per box)"** strategy (`watermark_remover/doc_detect.py`).


In [ ]:
import pathlib, shutil
import torch
from IPython.display import FileLink, display

best_src = ALPHA_RUN_DIR / 'best.pt'
final_path = pathlib.Path('/kaggle/working/alpha_net_best_final.pt')

if best_src.exists():
    shutil.copy2(best_src, final_path)
    ckpt = torch.load(final_path, map_location='cpu', weights_only=True)
    size_mb = final_path.stat().st_size / 1e6
    print('epoch:', ckpt['epoch'])
    print('val metrics:', ckpt['val_metrics'])
    print(f'size: {size_mb:.2f} MB')
    display(FileLink(final_path.name))
else:
    print(f'No best.pt found at {best_src} -- run the training cell (10.7) first.')
